In [1]:
!pip install captum
!pip install jsonargparse
!git clone https://github.com/pokaxpoka/deep_Mahalanobis_detector.git
import sys
sys.path.append('./deep_Mahalanobis_detector')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 7.9 MB/s eta 0:00:00
Cloning into 'deep_Mahalanobis_detector'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 72 (delta 8), reused 4 (delta 4), pack-reused 54 (from 1)
Receiving objects: 100% (72/72), 46.50 KiB | 4.23 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [2]:
!pip install streamlit
!pip install streamlit localtunnel
!pip install imutils
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 80.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 107.7 MB/s eta 0:00:0000:01
ERROR: Could not find a version that satisfies the requirement localtunnel (from versions: none)
ERROR: No matching distribution found for localtunnel
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴npm notice
npm notice New major version of npm available! 10.8.2 -> 11.4.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.4.1
npm notice To update run: npm install -g npm@11.4.1
npm notice
⠴

In [3]:
%%writefile app_temp.py

from __future__ import print_function
import numpy as np
import torch
import random
from typing import Callable, Any, List, Union
from tqdm import tqdm
import math
import os
import warnings
import tempfile
import logging
warnings.filterwarnings("ignore")
from itertools import product
from typing import Any, Callable, Dict
from captum.attr import GuidedBackprop, InputXGradient, IntegratedGradients, NoiseTunnel
from jsonargparse import CLI
from pytorch_lightning import LightningDataModule
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision.transforms import Normalize
import torchvision.transforms as torch_trans
from torchvision.datasets import CIFAR10, SVHN
from torchvision import transforms
import pytorch_lightning as pl
from torch.nn import Conv2d, Flatten, Linear, MaxPool2d, ReLU, Sequential, Unflatten
from torchmetrics import Accuracy, AUROC, CatMetric
from torchmetrics.utilities.data import dim_zero_cat
from torch import is_tensor
import torch.nn.functional as F

import torch.nn as nn
from torch.autograd import Variable

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from jsonargparse import CLI

from tqdm.auto import tqdm as tqdm_auto

from deep_Mahalanobis_detector import models, data_loader
from sklearn.base import TransformerMixin, BaseEstimator, clone
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import precision_recall_curve, roc_auc_score, accuracy_score, auc
from sklearn.covariance import EmpiricalCovariance
from sklearn.linear_model import LogisticRegressionCV
from sklearn.utils import shuffle
from sklearn.model_selection import PredefinedSplit
from sklearn.svm import OneClassSVM
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from skopt.callbacks import DeltaYStopper
from skopt import BayesSearchCV
from skopt.space import Real

from skimage.restoration import denoise_tv_chambolle, denoise_tv_bregman
from skimage.util import random_noise, img_as_float
from skimage import color

from scipy.optimize import minimize

import pickle as pkl
import json

from enum import Enum
import re
import pickle

from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import mahalanobis

import torchvision.utils as vutils

import torch.optim as optim
import torchvision
import time
from scipy import misc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, cdist, squareform

CROP_TYPE = ['center', 'random', 'sliding']

class Crop(object):
    def __init__(self, crop_type=None, crop_frac=1.0, sliding_crop_position=None):
        assert crop_frac <= 1.0, "crop_frac can't be greater than 1.0"
        if sliding_crop_position is not None:
            assert sliding_crop_position < 9

        assert (crop_type is None or crop_type in CROP_TYPE), ("{} is not a valid crop_type".format(crop_type))

        self.crop_type = crop_type
        self.crop_frac = crop_frac
        self.sliding_crop_position = sliding_crop_position
        
    def __call__(self, img):
        assert img is not None, "img should not be None"
        assert is_tensor(img), "Tensor expected"
        h = img.size(1)
        w = img.size(2)
        h2 = int(h * self.crop_frac)
        w2 = int(w * self.crop_frac)
        h_range = h - h2
        w_range = w - w2

        if self.crop_type == 'sliding':
            assert self.sliding_crop_position is not None
            row = int(self.sliding_crop_position / 3)
            col = self.sliding_crop_position % 3
            x = col * int(w_range / 2)
            y = row * int(h_range / 2)

        elif self.crop_type == 'random':
            x, y = random.randint(0, w_range), random.randint(0, h_range)

        elif self.crop_type == 'center':
            y = int(h_range / 2)
            x = int(w_range / 2)

        if self.crop_type is not None:
            img = img.narrow(1, y, h2).narrow(2, x, w2).clone()

        return img
    def update_sliding_position(self, sliding_crop_position):
        assert sliding_crop_position >= 0 and sliding_crop_position < 9, "Only 9 sliding positions supported"
        self.sliding_crop_position = sliding_crop_position

class Scale(object):
    def __init__(self, size, mean_std=None):
        if mean_std is not None:
            assert 'MEAN' in mean_std
            assert 'STD' in mean_std
        self.size = size
        self.mean_std = mean_std
        
    def __call__(self, img):
        
        assert img is not None, "img should not be None"
        assert is_tensor(img), "Tensor expected"

        if not img.size(1) == self.size:
            if self.mean_std:
                img = Unnormalize(mean=self.mean_std['MEAN'], std=self.mean_std['STD'])(img)
            img = torch_trans.ToPILImage()(img)
            img = torch_trans.Scale(self.size)(img)
            img = torch_trans.ToTensor()(img)
            if self.mean_std:
                img = torch_trans.Normalize(mean=self.mean_std['MEAN'], std=self.mean_std['STD'])(img)

        return img

class Normalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std
    def __call__(self, imgs):
        
        assert imgs is not None, "img should not be None"
        assert is_tensor(imgs), "Tensor expected"
        imgs_trans = imgs.clone()
        if len(imgs.size()) == 3:
            for i in range(imgs.size(0)):
                imgs_trans[i, :, :] = (imgs_trans[i, :, :] - self.mean[i]) / self.std[i]
        else:
            for i in range(imgs.size(1)):
                imgs_trans[:, i, :, :] = ((imgs_trans[:, i, :, :] - self.mean[i]) / self.std[i])
        return imgs_trans

CHECKPOINT_FILE = 'checkpoint.torch'

def get_probs(model, imgs, output_prob=False):
    softmax = torch.nn.Softmax(1)
    # probs = torch.zeros(imgs.size(0), n_classes)
    imgsvar = torch.autograd.Variable(imgs.squeeze(), volatile=True)
    output = model(imgsvar)
    if output_prob:
        probs = output.data.cpu()
    else:
        probs = softmax.forward(output).data.cpu()

    return probs

def get_labels(model, input, output_prob=False):
    probs = get_probs(model, input, output_prob)
    _, label = probs.max(1)
    return label.squeeze()

def tv(x, p):
    f = np.linalg.norm(x[1:, :] - x[:-1, :], p, axis=1).sum()
    f += np.linalg.norm(x[:, 1:] - x[:, :-1], p, axis=0).sum()
    return f

def tv_dx(x, p):
    if p == 1:
        x_diff0 = np.sign(x[1:, :] - x[:-1, :])
        x_diff1 = np.sign(x[:, 1:] - x[:, :-1])
    elif p > 1:
        x_diff0_norm = np.power(np.linalg.norm(x[1:, :] - x[:-1, :], p, axis=1), p - 1)
        x_diff1_norm = np.power(np.linalg.norm(x[:, 1:] - x[:, :-1], p, axis=0), p - 1)
        x_diff0_norm[x_diff0_norm < 1e-3] = 1e-3
        x_diff1_norm[x_diff1_norm < 1e-3] = 1e-3
        x_diff0_norm = np.repeat(x_diff0_norm[:, np.newaxis], x.shape[1], axis=1)
        x_diff1_norm = np.repeat(x_diff1_norm[np.newaxis, :], x.shape[0], axis=0)
        x_diff0 = p * np.power(x[1:, :] - x[:-1, :], p - 1) / x_diff0_norm
        x_diff1 = p * np.power(x[:, 1:] - x[:, :-1], p - 1) / x_diff1_norm
    df = np.zeros(x.shape)
    df[:-1, :] = -x_diff0
    df[1:, :] += x_diff0
    df[:, :-1] -= x_diff1
    df[:, 1:] += x_diff1
    return df

def tv_l2(x, y, w, lam, p):
    f = 0.5 * np.power(x - y.flatten(), 2).dot(w.flatten())
    x = np.reshape(x, y.shape)
    return f + lam * tv(x, p)

def tv_12_dx(x, y, w, lam, p):
    x = np.reshape(x, y.shape)
    df = (x - y) * w
    return df.flatten() + lam * tv_dx(x, p).flatten()

def tv_inf(x, y, lam, p, tau):
    x = np.reshape(x, y.shape)
    return tau + lam * tv(x, p)

def tv_inf_dx(x, y, lam, p, tau):
    x = np.reshape(x, y.shape)
    return lam * tv_dx(x, p).flatten()


class FGSMode(Enum):
    CARLINI = 'carlini'
    LOGIT = 'logit'
    @classmethod
    def has_value(cls, value):
        return (any(value == item.value for item in cls))
    def __str__(self):
        return str(self.value)

def compute_stats(all_inputs, all_outputs, status):
    
    ssim = None
    all_inputs = all_inputs.view(all_inputs.size(0), -1)
    all_outputs = all_outputs.view(all_outputs.size(0), -1)
    diff = (all_inputs - all_outputs).norm(2, 1).squeeze()
    diff = diff.div(all_inputs.norm(2, 1).squeeze())
    n_succ = status.eq(1).sum()
    n_fail = status.eq(-1).sum()
    return (diff.mean(), ssim, float(n_succ) / float(n_succ + n_fail))

def fgs(model, input, target, step_size=0.1, train_mode=False, mode=None, verbose=True):
    
    is_gpu = next(model.parameters()).is_cuda
    if mode:
        assert FGSMode.has_value(mode)
    if train_mode:
        model.train()
    else:
        model.eval()
    model.zero_grad()
    input_var = torch.autograd.Variable(input, requires_grad=True)
    output = model(input_var)
    if is_gpu:
        cpu_targets = target.clone()
        target = target.cuda(non_blocking=True)
    else:
        cpu_targets = target
    target_var = torch.autograd.Variable(target)
    _, pred = output.data.cpu().max(1)
    pred = pred.squeeze()
    corr = pred.eq(cpu_targets)
    if mode == str(FGSMode.CARLINI):
        output = output.mul(-1).add(1).log()
        criterion = torch.nn.NLLLoss()
    elif mode == str(FGSMode.LOGIT):
        criterion = torch.nn.NLLLoss()
    else:
        criterion = torch.nn.CrossEntropyLoss()
    if is_gpu:
        criterion = criterion.cuda()
    loss = criterion(output, target_var)
    loss.backward()
    grad_sign = input_var.grad.sign()
    input_var2 = input_var + step_size * grad_sign
    output2 = model(input_var2)
    _, pred2 = output2.data.cpu().max(1)
    pred2 = pred2.squeeze()
    status = torch.zeros(input_var.size(0)).long()
    status[corr] = 2 * pred[corr].ne(pred2[corr]).long() - 1
    return (status, step_size * grad_sign.data.cpu())

def fgs_search(model, input, target, r, rb=0.9, precision=2, a=-10.0, b=0.0, batch_size=25, verbose=True):
    opt_exp = b
    for i in range(precision):
        # search through predefined range on first iteration
        if i == 0:
            lower = a
        else:
            lower = opt_exp - pow(10, 1 - i)
        exponents = torch.arange(lower, opt_exp, pow(10, -i))
        for exponent in exponents:
            step_size = pow(2, exponent)
            succ = torch.zeros(input.size(0)).byte()
            dataset = torch.utils.data.TensorDataset(input + step_size * r, target)
            dataloader = torch.utils.data.DataLoader(
                dataset, batch_size=batch_size, shuffle=False, num_workers=4)
            count = 0
            for x, y in dataloader:
                x_batch = torch.autograd.Variable(x).cuda()
                output = model.forward(x_batch)
                _, pred = output.data.max(1)
                pred = pred.squeeze().cpu()
                succ[count:(count + x.size(0))] = pred.ne(y)
                count = count + x.size(0)
            success_rate = succ.float().mean()
            if verbose:
                print('step size = %1.4f,  success rate = %1.4f'
                      % (step_size, success_rate))
            if success_rate >= rb:
                opt_exp = exponent
                break
    return (succ, pow(2, opt_exp))

def fgs_compute_status(model, inputs, outputs, targets, status, batch_size=25, threshold=0.9, verbose=True):
    all_idx = torch.arange(0, status.size(0)).long()
    corr = all_idx[status.ne(0)]
    r = outputs - inputs
    succ, eps = fgs_search(
        model, inputs[corr], targets[corr], r[corr], rb=threshold,
        batch_size=batch_size, verbose=verbose)
    succ = succ.long()
    status[corr] = 2 * succ - 1
    return (status, eps)

def ifgs(model, input, target, max_iter=10, step_size=0.01, train_mode=False, mode=None, verbose=True):
    if train_mode:
        model.train()
    else:
        model.eval()
    pred = get_labels(model, input)
    corr = pred.eq(target)
    r = torch.zeros(input.size())
    for _ in range(max_iter):
        _, ri = fgs(model, input, target, step_size, train_mode, mode, verbose=verbose)
        r = r + ri
        input = input + ri
    pred_xp = get_labels(model, input + r)
    status = torch.zeros(input.size(0)).long()
    status[corr] = 2 * pred[corr].ne(pred_xp[corr]).long() - 1
    return (status, r)

def deepfool_single(model, imgs, target, n_classes, train_mode, max_iter=50, step_size=0.1, batch_size=10, labels=None, verbose=False):
    is_gpu = next(model.parameters()).is_cuda
    if train_mode:
        model.train()
    else:
        model.eval()
    cpu_targets = target
    imgs_var = torch.autograd.Variable(imgs)
    imgs_var2 = imgs_var.clone()
    r = torch.zeros(imgs_var.size()).cuda()
    criterion = torch.nn.NLLLoss()
    if is_gpu:
        criterion = criterion.cuda()
    for m in range(max_iter):
        imgs_var_in = imgs_var2.expand(1, imgs_var2.size(0), imgs_var2.size(1), imgs_var2.size(2))
        grad_input = imgs_var_in.repeat(n_classes, 1, 1, 1)
        output = model(imgs_var_in).clone()
        model.zero_grad()
        idx = torch.arange(0, n_classes).long()
        imgs_var_batch = torch.autograd.Variable(imgs_var_in.data.repeat(n_classes, 1, 1, 1), requires_grad=True)
        output_batch = model(imgs_var_batch)
        if is_gpu:
            _idx = idx.clone().cuda()
        else:
            _idx = idx.clone()
        loss_batch = criterion(output_batch, torch.autograd.Variable(_idx))
        loss_batch.backward()
        grad_input.index_copy_(0, torch.autograd.Variable(_idx), -imgs_var_batch.grad)
        f = (output - output[0][target].expand_as(output))
        w = grad_input - grad_input[target].expand_as(grad_input)
        w_norm = w.view(n_classes, -1).norm(2, 1)
        ratio = torch.abs(f).div(w_norm).data
        ratio[0][target] = float('inf')
        min_ratio, min_idx = ratio.min(1)
        min_w = w[min_idx[0]]
        min_norm = w_norm[min_idx[0]].data
        min_ratio = min_ratio[0]
        #min_norm = min_norm[0]
        min_norm = min_norm.item()
        ri = min_ratio / min_norm * step_size * min_w
        imgs_var2 = imgs_var2.add(ri)
        r = r.add(ri.data)
        imgs_var_in = imgs_var2.clone().expand(1, imgs_var2.size(0), imgs_var2.size(1), imgs_var2.size(2))
        output2 = model.forward(imgs_var_in).clone()
        _, pred2 = output2.data.cpu().max(1)
        #pred2 = pred2.squeeze()[0]
        pred2 = pred2.squeeze().item()
        diff = torch.norm(imgs_var - imgs_var2) / torch.norm(imgs_var)
        #diff = diff.data[0]
        diff = diff.data.item()
        if verbose:
            print('iteration ' + str(m + 1) +
                  ': perturbation norm ratio = ' + str(diff))
        if pred2 != cpu_targets:
            if verbose:
                if labels:
                    print('old label = %s, new label = %s' % (labels[cpu_targets],
                                                                labels[pred2]))
                else:
                    print('old label = %d, new label = %d' % (cpu_targets, pred2))
            break
    return (pred2 != target, imgs_var_in.data)

def deepfool(model, input, target, n_classes, train_mode=False, max_iter=5, step_size=0.1, batch_size=25, labels=None):
    pred = get_labels(model, input, batch_size)
    status = torch.zeros(input.size(0)).long()
    r = torch.zeros(input.size())
    for i in range(input.size(0)):
        status[i], r[i] = deepfool_single(
            model, input[i], target[i], n_classes, train_mode,
            max_iter, step_size, batch_size, labels)
    status = 2 * status - 1
    status[pred.ne(target)] = 0
    return (status, r)

def cw(model, input, target, weight, loss_str, bound=0, tv_weight=0, max_iter=100, step_size=0.01, kappa=0, p=2, crop_frac=1.0, drop_rate=0.0, train_mode=False, verbose=False):
    is_gpu = next(model.parameters()).is_cuda
    if train_mode:
        model.train()
    else:
        model.eval()
    pred = get_labels(model, input)
    corr = pred.eq(target)
    w = torch.autograd.Variable(input, requires_grad=True)
    best_w = torch.Tensor(input.size())
    best_w.copy_(input)
    best_loss = float('inf')
    optimizer = torch.optim.Adam([w], lr=step_size)
    input_var = torch.autograd.Variable(input)
    input_vec = input.view(input.size(0), -1)
    to_pil = torch_trans.ToPILImage()
    scale_up = torch_trans.Resize((w.size(2), w.size(3)))
    scale_down = torch_trans.Resize((int(crop_frac * w.size(2)), int(crop_frac * w.size(3))))
    to_tensor = torch_trans.ToTensor()
    probs = get_probs(model, input)
    _, top2 = probs.topk(2, 1)
    argmax = top2[:, 0]
    for j in range(top2.size(0)):
        if argmax[j] == target[j]:
            argmax[j] = top2[j, 1]
    for i in range(max_iter):
        if i > 0:
            w.grad.data.fill_(0)
        model.zero_grad()
        if loss_str == 'l2':
            loss = torch.pow(w - input_var, 2).sum()
        elif loss_str == 'linf':
            loss = torch.clamp((w - input_var).abs() - bound, min=0).sum()
        else:
            raise ValueError('Unsupported loss: %s' % loss_str)
        #recons_loss = loss.data[0]
        recons_loss = loss.data.item()

        w_data = w.data
        if crop_frac < 1 and i % 3 == 1:
            w_cropped = torch.zeros(
                w.size(0), w.size(1), int(crop_frac * w.size(2)),
                int(crop_frac * w.size(3)))
            locs = torch.zeros(w.size(0), 4).long()
            w_in = torch.zeros(w.size())
            for m in range(w.size(0)):
                locs[m] = torch.LongTensor(Crop('random', crop_frac)(w_data[m]))
                w_cropped = w_data[m, :, locs[m][0]:(locs[m][0] + locs[m][2]),
                                   locs[m][1]:(locs[m][1] + locs[m][3])]
                minimum = w_cropped.min()
                maximum = w_cropped.max() - minimum
                w_in[m] = to_tensor(scale_up(to_pil((w_cropped - minimum) / maximum)))
                w_in[m] = w_in[m] * maximum + minimum
            w_in = torch.autograd.Variable(w_in, requires_grad=True)
        else:
            w_in = torch.autograd.Variable(w_data, requires_grad=True)
        if drop_rate == 0 and i % 3 == 2:
            output = model.forward(w_in)
        else:
            output = model.forward(torch.nn.Dropout(p=drop_rate).forward(w_in))
        for j in range(output.size(0)):
            loss += weight * torch.clamp(
                output[j][target[j]] - output[j][argmax[j]] + kappa, min=0)
        #adv_loss = loss.data[0] - recons_loss
        adv_loss = loss.data.item() - recons_loss
        if is_gpu:
            loss = loss.cuda()
        loss.backward()
        if crop_frac < 1 and i % 3 == 1:
            grad_full = torch.zeros(w.size())
            grad_cpu = w_in.grad.data
            for m in range(w.size(0)):
                minimum = grad_cpu[m].min()
                maximum = grad_cpu[m].max() - minimum
                grad_m = to_tensor(scale_down(
                    to_pil((grad_cpu[m] - minimum) / maximum)))
                grad_m = grad_m * maximum + minimum
                grad_full[m, :, locs[m][0]:(locs[m][0] + locs[m][2]),
                          locs[m][1]:(locs[m][1] + locs[m][3])] = grad_m
            w.grad.data.add_(grad_full)
        else:
            w.grad.data.add_(w_in.grad.data)
        w_cpu = w.data.cpu().numpy()
        input_np = input.cpu().numpy()
        tv_loss = 0
        if tv_weight > 0:
            for j in range(output.size(0)):
                for k in range(3):
                    tv_loss += tv_weight * minimize_tv.tv(
                        w_cpu[j, k] - input_np[j, k], p)
                    grad = tv_weight * torch.from_numpy(
                        minimize_tv.tv_dx(w_cpu[j, k] - input_np[j, k], p))
                    w.grad.data[j, k].add_(grad.float())
        optimizer.step()
        #total_loss = loss.data.cpu()[0] + tv_loss
        total_loss = loss.data.cpu().item() + tv_loss
        # w.data = utils.img_to_tensor(utils.transform_img(w.data), scale=False)
        output_vec = w.data
        preds = get_labels(model, output_vec)
        output_vec = output_vec.view(output_vec.size(0), -1)
        diff = (input_vec - output_vec).norm(2, 1).squeeze()
        diff = diff.div(input_vec.norm(2, 1).squeeze())
        rb = diff.mean()
        sr = float(preds.ne(target).sum()) / target.size(0)
        if verbose:
            print('iteration %d: loss = %f, %s_loss = %f, ' 'adv_loss = %f, tv_loss = %f' % (i + 1, total_loss, loss_str, recons_loss, adv_loss, tv_loss))
            print('robustness = %f, success rate = %f' % (rb, sr))
        if total_loss < best_loss:
            best_loss = total_loss
            best_w = w.data.clone()
    pred_xp = get_labels(model, best_w)
    status = torch.zeros(input.size(0)).long()
    status[corr] = 2 * pred[corr].ne(pred_xp[corr]).long() - 1
    return (status, best_w)

class MahalanobisLoss(nn.Module):
    def __init__(self):
        super(_MahalanobisLoss, self).__init__()
    def forward(self, feature, mean, inverse_cov):
        mahalaonbis_loss = 0            
        zero_f = feature - mean
        mahalaonbis_loss = torch.mm(torch.mm(zero_f, inverse_cov), zero_f.t()).diag()
        mahalaonbis_loss = torch.mean(mahalaonbis_loss)
        return mahalaonbis_loss

class _MahalanobisEnsembleLoss(nn.Module):
    def __init__(self):
        super(_MahalanobisEnsembleLoss, self).__init__()
        
    def forward(self, feature, mean, inverse_cov, weight, top2_index):
        mahalaonbis_loss = 0
        for i in range(len(mean)):
            temp_loss = 0
            final_mean = mean[i].index_select(0, top2_index.cuda())
            final_mean = Variable(final_mean)
            zero_f = feature[i] - final_mean
            temp_loss = torch.mm(torch.mm(zero_f, Variable(inverse_cov[i])), zero_f.t()).diag()
            mahalaonbis_loss += weight[i]*torch.mean(temp_loss)
        return mahalaonbis_loss

def attacking_mahalanobis(model, input, target, weight, loss_str, sample_mean, inv_sample_conv, index, bound=0, max_iter=100, step_size=0.01, kappa=0, p=2, drop_rate=0.0,
       train_mode=False, verbose=False):
    is_gpu = next(model.parameters()).is_cuda
    if train_mode:
        model.train()
    else:
        model.eval()
        
    pred = get_labels(model, input)
    corr = pred.eq(target)
    w = torch.autograd.Variable(input, requires_grad=True)
    best_w = torch.Tensor(input.size())
    best_w.copy_(input)
    best_loss = float('inf')
    optimizer = torch.optim.Adam([w], lr=step_size)
    input_var = torch.autograd.Variable(input)
    input_vec = input.view(input.size(0), -1)
    
    probs = get_probs(model, input)
    _, top2 = probs.topk(2, 1)
    argmax = top2[:, 0]
    top2_index = top2[:, 1]
    for j in range(top2.size(0)):
        if argmax[j] == target[j]:
            argmax[j] = top2[j, 1]
    
    mahalanobis_criterion = _MahalanobisLoss().cuda()
    total_mean = sample_mean[index]
    final_mean = total_mean.index_select(0, top2_index.cuda())
    final_mean = Variable(final_mean)
    
    print(final_mean.size())
    final_covariance = Variable(inv_sample_conv[index])
    
    for i in range(max_iter):
        if i > 0:
            w.grad.data.fill_(0)
        model.zero_grad()
        
        # Norm constraints
        if loss_str == 'l2':
            loss = torch.pow(w - input_var, 2).sum()
        elif loss_str == 'linf':
            loss = torch.clamp((w - input_var).abs() - bound, min=0).sum()
        else:
            raise ValueError('Unsupported loss: %s' % loss_str)
        recons_loss = loss.data[0]
        
        # Adversarial loss
        w_data = w.data
        w_in = torch.autograd.Variable(w_data, requires_grad=True)
        output, out_features = model.feature_list(w_in)
        for j in range(len(out_features)):
            out_features[j] = out_features[j].view(out_features[j].size(0), out_features[j].size(1), -1)
            out_features[j] = torch.mean(out_features[j], 2)
        
        for j in range(output.size(0)):
            loss += weight * torch.clamp(
                output[j][target[j]] - output[j][argmax[j]] + kappa, min=0)
        OOD_out = out_features[index]
        temp_loss = mahalanobis_criterion(OOD_out, final_mean, final_covariance)
        
        #adv_loss = loss.data[0] - recons_loss
        adv_loss = loss.data.item() - recons_loss
        if is_gpu:
            loss = loss.cuda()
        loss.backward()
        
        w.grad.data.add_(w_in.grad.data)
        w_cpu = w.data.cpu().numpy()
        input_np = input.cpu().numpy()

        optimizer.step()
        total_loss = loss.data.cpu()[0]
        # w.data = utils.img_to_tensor(utils.transform_img(w.data), scale=False)
        output_vec = w.data
        preds = get_labels(model, output_vec)
        output_vec = output_vec.view(output_vec.size(0), -1)
        diff = (input_vec - output_vec).norm(2, 1).squeeze()
        diff = diff.div(input_vec.norm(2, 1).squeeze())
        rb = diff.mean()
        sr = float(preds.ne(target).sum()) / target.size(0)
        if verbose:
            print('iteration %d: loss = %f, %s_loss = %f, ''adv_loss = %f' % (i + 1, total_loss, loss_str, recons_loss, adv_loss))
            print('robustness = %f, success rate = %f' % (rb, sr))
        if total_loss < best_loss:
            best_loss = total_loss
            best_w = w.data.clone()
    pred_xp = get_labels(model, best_w)
    status = torch.zeros(input.size(0)).long()
    status[corr] = 2 * pred[corr].ne(pred_xp[corr]).long() - 1
    return (status, best_w)

def attacking_mahalanobis_all(model, input, target, weight, loss_str, sample_mean, inv_sample_conv, ensemble_weight, bound=0, max_iter=100, step_size=0.01, kappa=0, p=2, drop_rate=0.0, train_mode=False, verbose=False):
    is_gpu = next(model.parameters()).is_cuda
    if train_mode:
        model.train()
    else:
        model.eval()
        
    pred = get_labels(model, input)
    corr = pred.eq(target)
    w = torch.autograd.Variable(input, requires_grad=True)
    best_w = torch.Tensor(input.size())
    best_w.copy_(input)
    best_loss = float('inf')
    optimizer = torch.optim.Adam([w], lr=step_size)
    input_var = torch.autograd.Variable(input)
    input_vec = input.view(input.size(0), -1)
    
    probs = get_probs(model, input)
    _, top2 = probs.topk(2, 1)
    argmax = top2[:, 0]
    top2_index = top2[:, 1]
    for j in range(top2.size(0)):
        if argmax[j] == target[j]:
            argmax[j] = top2[j, 1]
    
    mahalanobis_criterion = _MahalanobisEnsembleLoss().cuda()
    print('attack all')
    for i in range(max_iter):
        if i > 0:
            w.grad.data.fill_(0)
        model.zero_grad()
        
        # Norm constraints
        if loss_str == 'l2':
            loss = torch.pow(w - input_var, 2).sum()
        elif loss_str == 'linf':
            loss = torch.clamp((w - input_var).abs() - bound, min=0).sum()
        else:
            raise ValueError('Unsupported loss: %s' % loss_str)
        recons_loss = loss.data[0]
        
        # Adversarial loss
        w_data = w.data
        w_in = torch.autograd.Variable(w_data, requires_grad=True)
        output, out_features = model.feature_list(w_in)
        for j in range(len(out_features)):
            out_features[j] = out_features[j].view(out_features[j].size(0), out_features[j].size(1), -1)
            out_features[j] = torch.mean(out_features[j], 2)
        
        for j in range(output.size(0)):
            loss += weight * torch.clamp(
                output[j][target[j]] - output[j][argmax[j]] + kappa, min=0)
        temp_loss = mahalanobis_criterion(out_features, sample_mean, inv_sample_conv, ensemble_weight, top2_index)
        
        #adv_loss = loss.data[0] - recons_loss
        adv_loss = loss.data.item() - recons_loss
        if is_gpu:
            loss = loss.cuda()
        loss.backward()
        
        w.grad.data.add_(w_in.grad.data)
        w_cpu = w.data.cpu().numpy()
        input_np = input.cpu().numpy()

        optimizer.step()
        total_loss = loss.data.cpu()[0]
        # w.data = utils.img_to_tensor(utils.transform_img(w.data), scale=False)
        output_vec = w.data
        preds = get_labels(model, output_vec)
        output_vec = output_vec.view(output_vec.size(0), -1)
        diff = (input_vec - output_vec).norm(2, 1).squeeze()
        diff = diff.div(input_vec.norm(2, 1).squeeze())
        rb = diff.mean()
        sr = float(preds.ne(target).sum()) / target.size(0)
        if verbose:
            print('iteration %d: loss = %f, %s_loss = %f, ''adv_loss = %f' % (i + 1, total_loss, loss_str, recons_loss, adv_loss))
            print('robustness = %f, success rate = %f' % (rb, sr))
        if total_loss < best_loss:
            best_loss = total_loss
            best_w = w.data.clone()
    pred_xp = get_labels(model, best_w)
    status = torch.zeros(input.size(0)).long()
    status[corr] = 2 * pred[corr].ne(pred_xp[corr]).long() - 1
    return (status, best_w)

def rand_sign(model, input, target, step_size, num_bins=100):
    x = torch.autograd.Variable(input, requires_grad=True)
    output = model.forward(x)
    _, pred = output.data.max(1)
    pred = pred.squeeze().cpu()
    corr = pred.eq(target)
    target = torch.autograd.Variable(target)
    P = torch.ones(input.size(0), num_bins)
    sign = 2 * torch.bernoulli(P) - 1
    H = torch.rand(input.size())
    H = (H * num_bins).floor().int()
    r = torch.zeros(input.size())
    for i in range(input.size(0)):
        for j in range(num_bins):
            r[i][H[i].eq(j)] = sign[i, j]
    xp = x + step_size * torch.autograd.Variable(r.cuda())
    output_xp = model.forward(xp).clone()
    _, pred_xp = output_xp.data.max(1)
    pred_xp = pred_xp.squeeze().cpu()
    status = torch.zeros(x.size(0)).long()
    status[corr] = 2 * pred[corr].ne(pred_xp[corr]).long() - 1
    return (status, step_size * r)

def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )
            
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class PreActBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(PreActBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False))
    
    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        out += shortcut
        return out

class Bottleneck(nn.Module):
    expansion=4

    def __init__(self, in_planes, planes, stride=1):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion*planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion*planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class PreActBottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(PreActBottleneck, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion*planes, kernel_size=1, bias=False)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False))
    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.conv3(F.relu(self.bn3(out)))
        out += shortcut
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = conv3x3(3,64)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512*block.expansion, num_classes)
        
    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)
        
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        y = self.linear(out)
        return y
        
    def feature_list(self, x):
        out_list = []
        out = F.relu(self.bn1(self.conv1(x)))
        out_list.append(out)
        out = self.layer1(out)
        out_list.append(out)
        out = self.layer2(out)
        out_list.append(out)
        out = self.layer3(out)
        out_list.append(out)
        out = self.layer4(out)
        out_list.append(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        y = self.linear(out)
        return y, out_list
        
    def intermediate_forward(self, x, layer_index):
        out = F.relu(self.bn1(self.conv1(x)))
        if layer_index == 1:
            out = self.layer1(out)
        elif layer_index == 2:
            out = self.layer1(out)
            out = self.layer2(out)
        elif layer_index == 3:
            out = self.layer1(out)
            out = self.layer2(out)
            out = self.layer3(out)
        elif layer_index == 4:
            out = self.layer1(out)
            out = self.layer2(out)
            out = self.layer3(out)
            out = self.layer4(out)               
        return out
        
    def penultimate_forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        penultimate = self.layer4(out)
        out = F.avg_pool2d(penultimate, 4)
        out = out.view(out.size(0), -1)
        y = self.linear(out)
        return y, penultimate

def ResNet18(num_c):
    return ResNet(PreActBlock, [2,2,2,2], num_classes=num_c)

def ResNet34(num_c):
    return ResNet(BasicBlock, [3,4,6,3], num_classes=num_c)

def test():
    net = ResNet18()
    y = net(Variable(torch.randn(1,3,32,32)))
    print(y.size())

def getSVHN(batch_size, TF, data_root='/kaggle/working', train=True, val=True, **kwargs):
    data_root = os.path.expanduser(os.path.join(data_root, 'svhn-data'))
    num_workers = kwargs.setdefault('num_workers', 1)
    kwargs.pop('input_size', None)
    def target_transform(target):
        new_target = target - 1
        if new_target == -1:
            new_target = 9
        return new_target

    ds = []
    if train:
        train_loader = torch.utils.data.DataLoader(
            datasets.SVHN(
                root=data_root, split='train', download=True,
                transform=TF,
            ),
            batch_size=batch_size, shuffle=True, **kwargs)
        ds.append(train_loader)

    if val:
        test_loader = torch.utils.data.DataLoader(
            datasets.SVHN(
                root=data_root, split='test', download=True,
                transform=TF,
            ),
            batch_size=batch_size, shuffle=False, **kwargs)
        ds.append(test_loader)
    ds = ds[0] if len(ds) == 1 else ds
    return ds

def getCIFAR10(batch_size, TF, data_root='/kaggle/working', train=True, val=True, **kwargs):
    data_root = os.path.expanduser(os.path.join(data_root, 'cifar10-data'))
    num_workers = kwargs.setdefault('num_workers', 1)
    kwargs.pop('input_size', None)
    ds = []
    if train:
        train_loader = torch.utils.data.DataLoader(
            datasets.CIFAR10(root=data_root, train=True, download=True, transform=TF),
            batch_size=batch_size, shuffle=True, **kwargs)
        ds.append(train_loader)
    if val:
        test_loader = torch.utils.data.DataLoader(
            datasets.CIFAR10(
                root=data_root, train=False, download=True,
                transform=TF),
            batch_size=batch_size, shuffle=False, **kwargs)
        ds.append(test_loader)
    ds = ds[0] if len(ds) == 1 else ds
    return ds

def getCIFAR100(batch_size, TF, data_root='/kaggle/working', train=True, val=True, **kwargs):
    data_root = os.path.expanduser(os.path.join(data_root, 'cifar100-data'))
    num_workers = kwargs.setdefault('num_workers', 1)
    kwargs.pop('input_size', None)
    ds = []
    if train:
        train_loader = torch.utils.data.DataLoader(datasets.CIFAR100(root=data_root, train=True, download=True, transform=TF), batch_size=batch_size, shuffle=True, **kwargs)
        ds.append(train_loader)

    if val:
        test_loader = torch.utils.data.DataLoader(
            datasets.CIFAR100(
                root=data_root, train=False, download=True,
                transform=TF),
            batch_size=batch_size, shuffle=False, **kwargs)
        ds.append(test_loader)
    ds = ds[0] if len(ds) == 1 else ds
    return ds

def getTargetDataSet(data_type, batch_size, input_TF, dataroot):
    if data_type == 'cifar10':
        train_loader, test_loader = getCIFAR10(batch_size=batch_size, TF=input_TF, data_root=dataroot, num_workers=1)
    elif data_type == 'cifar100':
        train_loader, test_loader = getCIFAR100(batch_size=batch_size, TF=input_TF, data_root=dataroot, num_workers=1)
    elif data_type == 'svhn':
        train_loader, test_loader = getSVHN(batch_size=batch_size, TF=input_TF, data_root=dataroot, num_workers=1)

    return train_loader, test_loader

def getNonTargetDataSet(data_type, batch_size, input_TF, dataroot):
    if data_type == 'cifar10':
        _, test_loader = getCIFAR10(batch_size=batch_size, TF=input_TF, data_root=dataroot, num_workers=1)
    elif data_type == 'svhn':
        _, test_loader = getSVHN(batch_size=batch_size, TF=input_TF, data_root=dataroot, num_workers=1)
    elif data_type == 'cifar100':
        _, test_loader = getCIFAR100(batch_size=batch_size, TF=input_TF, data_root=dataroot, num_workers=1)
    elif data_type == 'imagenet_resize':
        dataroot = os.path.expanduser(os.path.join(dataroot, 'Imagenet_resize'))
        testsetout = datasets.ImageFolder(dataroot, transform=input_TF)
        test_loader = torch.utils.data.DataLoader(testsetout, batch_size=batch_size, shuffle=False, num_workers=1)
    elif data_type == 'lsun_resize':
        dataroot = os.path.expanduser(os.path.join(dataroot, 'LSUN_resize'))
        testsetout = datasets.ImageFolder(dataroot, transform=input_TF)
        test_loader = torch.utils.data.DataLoader(testsetout, batch_size=batch_size, shuffle=False, num_workers=1)
    return test_loader

def maina():

    batch_size = 200  # Batch size for data loader
    dataset = 'cifar100'  # Change as needed: 'cifar10', 'cifar100', 'svhn'
    dataroot = '/kaggle/working'  # Path to dataset
    outf = '/kaggle/working/CIFAR100/adv_output/'  # Folder to output results
    num_classes = 10  # Number of classes
    net_type = 'resnet'  # Change as needed: 'resnet', 'densenet'
    gpu = 0  # GPU index
    adv_type = 'FGSM'  # Change as needed: 'FGSM', 'BIM', 'DeepFool', 'CWL2'
    
    # set the path to pre-trained model and output
    pre_trained_net = '/kaggle/input/resnet-pth/' + net_type + '_' + dataset + '.pth'
    outf = outf + net_type + '_' + dataset + '/'
    if os.path.isdir(outf) == False:
        os.makedirs(outf)
    torch.cuda.manual_seed(0)
    torch.cuda.set_device(gpu)
    # check the in-distribution dataset
    if dataset == 'cifar100':
        num_classes = 100
    if adv_type == 'FGSM':
        adv_noise = 0.05
    elif adv_type == 'BIM':
        adv_noise = 0.01
    elif adv_type == 'DeepFool':
        if net_type == 'resnet':
            if dataset == 'cifar10':
                adv_noise = 0.18
            elif dataset == 'cifar100':
                adv_noise = 0.03
            else:
                adv_noise = 0.1
        else:
            if dataset == 'cifar10':
                adv_noise = 0.6
            elif dataset == 'cifar100':
                adv_noise = 0.1
            else:
                adv_noise = 0.5

    if net_type == 'resnet':
        model = ResNet34(num_c=num_classes)
        model.load_state_dict(torch.load(pre_trained_net, map_location = "cuda:" + str(gpu)))
        in_transform = transforms.Compose([transforms.ToTensor(),
                                           transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),])
        
        min_pixel = -2.42906570435
        max_pixel = 2.75373125076
        if dataset == 'cifar10':
            if adv_type == 'FGSM':
                random_noise_size = 0.25 / 4
            elif adv_type == 'BIM':
                random_noise_size = 0.13 / 2
            elif adv_type == 'DeepFool':
                random_noise_size = 0.25 / 4
            elif adv_type == 'CWL2':
                random_noise_size = 0.05 / 2
        elif dataset == 'cifar100':
            if adv_type == 'FGSM':
                random_noise_size = 0.25 / 8
            elif adv_type == 'BIM':
                random_noise_size = 0.13 / 4
            elif adv_type == 'DeepFool':
                random_noise_size = 0.13 / 4
            elif adv_type == 'CWL2':
                random_noise_size = 0.05 / 2
        else:
            if adv_type == 'FGSM':
                random_noise_size = 0.25 / 4
            elif adv_type == 'BIM':
                random_noise_size = 0.13 / 2
            elif adv_type == 'DeepFool':
                random_noise_size = 0.126
            elif adv_type == 'CWL2':
                random_noise_size = 0.05 / 1 
            
    model.cuda()
    print('load model: ' + net_type)
    
    # load dataset
    print('load target data: ', dataset)
    _, test_loader = data_loader.getTargetDataSet(dataset, batch_size, in_transform, dataroot)
    
    print('Attack: ' + adv_type  +  ', Dist: ' + dataset + '\n')
    model.eval()
    adv_data_tot, clean_data_tot, noisy_data_tot = 0, 0, 0
    label_tot = 0
    
    correct, adv_correct, noise_correct = 0, 0, 0
    total, generated_noise = 0, 0

    criterion = nn.CrossEntropyLoss().cuda()

    selected_list = []
    selected_index = 0
    
    for data, target in test_loader:
        data, target = data.cuda(), target.cuda()
        data, target = Variable(data, volatile=True), Variable(target)
        output = model(data)

        # compute the accuracy
        pred = output.data.max(1)[1]
        equal_flag = pred.eq(target.data).cpu()
        correct += equal_flag.sum()

        noisy_data = torch.add(data.data, random_noise_size, torch.randn(data.size()).cuda()) 
        noisy_data = torch.clamp(noisy_data, min_pixel, max_pixel)

        if total == 0:
            clean_data_tot = data.clone().data.cpu()
            label_tot = target.clone().data.cpu()
            noisy_data_tot = noisy_data.clone().cpu()
        else:
            clean_data_tot = torch.cat((clean_data_tot, data.clone().data.cpu()),0)
            label_tot = torch.cat((label_tot, target.clone().data.cpu()), 0)
            noisy_data_tot = torch.cat((noisy_data_tot, noisy_data.clone().cpu()),0)
            
        # generate adversarial
        model.zero_grad()
        inputs = Variable(data.data, requires_grad=True)
        output = model(inputs)
        loss = criterion(output, target)
        loss.backward()

        if adv_type == 'FGSM': 
            gradient = torch.ge(inputs.grad.data, 0)
            gradient = (gradient.float()-0.5)*2
            if net_type == 'densenet':
                gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (63.0/255.0))
                gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (62.1/255.0))
                gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (66.7/255.0))
            else:
                gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (0.2023))
                gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (0.1994))
                gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (0.2010))

        elif adv_type == 'BIM': 
            gradient = torch.sign(inputs.grad.data)
            for k in range(5):
                inputs = torch.add(inputs.data, adv_noise, gradient)
                inputs = torch.clamp(inputs, min_pixel, max_pixel)
                inputs = Variable(inputs, requires_grad=True)
                output = model(inputs)
                loss = criterion(output, target)
                loss.backward()
                gradient = torch.sign(inputs.grad.data)
                if net_type == 'densenet':
                    gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (63.0/255.0))
                    gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (62.1/255.0))
                    gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (66.7/255.0))
                else:
                    gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (0.2023))
                    gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (0.1994))
                    gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (0.2010))

        if adv_type == 'DeepFool':
            _, adv_data = deepfool(model, data.data.clone(), target.data.cpu(), num_classes, step_size=adv_noise, train_mode=False)
            adv_data = adv_data.cuda()
        elif adv_type == 'CWL2':
            _, adv_data = cw(model, data.data.clone(), target.data.cpu(), 1.0, 'l2', crop_frac=1.0)
        else:
            adv_data = torch.add(inputs.data, adv_noise, gradient)
            
        adv_data = torch.clamp(adv_data, min_pixel, max_pixel)
        
        # measure the noise 
        temp_noise_max = torch.abs((data.data - adv_data).view(adv_data.size(0), -1))
        temp_noise_max, _ = torch.max(temp_noise_max, dim=1)
        generated_noise += torch.sum(temp_noise_max)

        if total == 0:
            flag = 1
            adv_data_tot = adv_data.clone().cpu()
        else:
            adv_data_tot = torch.cat((adv_data_tot, adv_data.clone().cpu()),0)

        output = model(Variable(adv_data, volatile=True))
        # compute the accuracy
        pred = output.data.max(1)[1]
        equal_flag_adv = pred.eq(target.data).cpu()
        adv_correct += equal_flag_adv.sum()
        
        output = model(Variable(noisy_data, volatile=True))
        # compute the accuracy
        pred = output.data.max(1)[1]
        equal_flag_noise = pred.eq(target.data).cpu()
        noise_correct += equal_flag_noise.sum()
        
        for i in range(data.size(0)):
            if equal_flag[i] == 1 and equal_flag_noise[i] == 1 and equal_flag_adv[i] == 0:
                selected_list.append(selected_index)
            selected_index += 1
            
        total += data.size(0)

    selected_list = torch.LongTensor(selected_list)
    clean_data_tot = torch.index_select(clean_data_tot, 0, selected_list)
    adv_data_tot = torch.index_select(adv_data_tot, 0, selected_list)
    noisy_data_tot = torch.index_select(noisy_data_tot, 0, selected_list)
    label_tot = torch.index_select(label_tot, 0, selected_list)

    torch.save(clean_data_tot, '%s/clean_data_%s_%s_%s.pth' % (outf, net_type, dataset, adv_type))
    torch.save(adv_data_tot, '%s/adv_data_%s_%s_%s.pth' % (outf, net_type, dataset, adv_type))
    torch.save(noisy_data_tot, '%s/noisy_data_%s_%s_%s.pth' % (outf, net_type, dataset, adv_type))
    torch.save(label_tot, '%s/label_%s_%s_%s.pth' % (outf, net_type, dataset, adv_type))

    print('Adversarial Noise:({:.2f})\n'.format(generated_noise / total))
    print('Final Accuracy: {}/{} ({:.2f}%)\n'.format(correct, total, 100. * correct / total))
    print('Adversarial Accuracy: {}/{} ({:.2f}%)\n'.format(adv_correct, total, 100. * adv_correct / total))
    print('Noisy Accuracy: {}/{} ({:.2f}%)\n'.format(noise_correct, total, 100. * noise_correct / total))

def load_model(model_name: str, dataset_name: str, maha_repo_dir: str, pretrained_dir: str, num_classes: int=10):
    sys.path.append(maha_repo_dir)

    model_path = os.path.join(pretrained_dir, f"{model_name}_{dataset_name}.pth")

    if model_name == "densenet":
        if dataset_name == "densenet":
            from deep_Mahalanobis_detector import models
            model = models.DenseNet3(100, num_classes)
            model.load_state_dict(torch.load(model_path))
        else:
            model = torch.load(f"{model_path}")
    elif model_name == "resnet":
        from deep_Mahalanobis_detector import models
        model = models.ResNet34(num_c=num_classes)
        model.load_state_dict(torch.load(model_path))

    else:
        raise ValueError()
    return model

class LabeledDataModule(LightningDataModule):
    def __init__(self, model_name: str, dataset_name: str, attack_name: str, labeled_dir: str, datasets_dir: str):
        self.model_name = model_name
        self.dataset_name = dataset_name
        self.attack_name = attack_name
        self.datasets_dir = datasets_dir
        self.labeled_dir = labeled_dir
    def setup(self, stage=None):
        self.setup_train_ds()
        self.setup_labeled_ds()
    def setup_train_ds(self):
        in_transforms = [transforms.ToTensor()]

        if self.model_name == "densenet":
            in_transforms.append(transforms.Normalize(
                    (125.3 / 255, 123.0 / 255, 113.9 / 255),
                    (63.0 / 255, 62.1 / 255.0, 66.7 / 255.0),))
        elif self.model_name == "resnet":
            in_transforms.append(transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)))
        else:
            raise ValueError()

        in_transforms = transforms.Compose(in_transforms)

        if self.dataset_name == "cifar10":
            self.datasets_dir = self.datasets_dir + '/CIFAR10'
            self.train_ds = CIFAR10(root=self.datasets_dir, train=True, transform=in_transforms, download=False)
        elif self.dataset_name == "svhn":
            self.datasets_dir = self.datasets_dir + '/SVHN'
            self.train_ds = SVHN(root=self.datasets_dir, split="train", transform=in_transforms, download=False)
        else:
            raise ValueError()
        
    def setup_labeled_ds(self):
        labeled_data_dirpath = os.path.join(self.labeled_dir, f"{self.dataset_name.upper()}")
        labeled_data_dirpath = os.path.join(labeled_data_dirpath, f"{self.attack_name}")

        setting_name = f"{self.model_name}_{self.dataset_name}_{self.attack_name}"

        self.labeled_clean_data = torch.load(os.path.join(labeled_data_dirpath, f"clean_data_{setting_name}.pth"))

        # Exclude last split for compatibility reasons
        self.split_size = 100 * (len(self.labeled_clean_data) // 100)
        self.labeled_clean_data = self.labeled_clean_data[: self.split_size]

        self.labeled_adv_data = torch.load(os.path.join(labeled_data_dirpath, f"adv_data_{setting_name}.pth"))[: self.split_size]
        self.labeled_noisy_data = torch.load(os.path.join(labeled_data_dirpath, f"noisy_data_{setting_name}.pth"))[: self.split_size]
        self.labeled_targets = torch.load(os.path.join(labeled_data_dirpath, f"label_{setting_name}.pth"))[: self.split_size]

        labeled_ds = TensorDataset(torch.cat((self.labeled_adv_data, self.labeled_clean_data, self.labeled_noisy_data,)), self.labeled_targets.tile((3,)))
        
        idxs_train, idxs_val, idxs_test = self.train_val_test_split_idxs(self.split_size)
        
        self.labeled_train_ds = Subset(labeled_ds, idxs_train)
        self.labeled_val_ds = Subset(labeled_ds, idxs_val)
        self.labeled_test_ds = Subset(labeled_ds, idxs_test)
        
    def train_dataloader(self, batch_size: int = 100):
        return DataLoader(self.train_ds, batch_size=batch_size)
    def labeled_train_dataloader(self, batch_size: int = 100):
        return DataLoader(self.labeled_train_ds, batch_size=batch_size)
    def labeled_val_dataloader(self, batch_size: int = 100):
        return DataLoader(self.labeled_val_ds, batch_size=batch_size)
    def labeled_test_dataloader(self, batch_size: int = 100):
        return DataLoader(self.labeled_test_ds, batch_size=batch_size)

    @staticmethod
    def train_val_test_split_idxs(split_size: int, trainval_frac: float = 0.1):
        train_val_span = int(split_size * trainval_frac)
        
        idxs_trainval = np.concatenate([np.arange(train_val_span), np.arange(split_size, split_size + train_val_span), np.arange(2 * split_size, 2 * split_size + train_val_span)])
        
        idxs_test = np.delete(np.arange(split_size * 3), idxs_trainval)
        pivot = int(len(idxs_trainval) / 6)
        
        idxs_train = np.concatenate(
            [
                idxs_trainval[:pivot],
                idxs_trainval[2 * pivot : 3 * pivot],
                idxs_trainval[4 * pivot : 5 * pivot],
            ])
        
        idxs_val = np.concatenate(
            [
                idxs_trainval[pivot : 2 * pivot],
                idxs_trainval[3 * pivot : 4 * pivot],
                idxs_trainval[5 * pivot :],
            ])
        
        return idxs_train, idxs_val, idxs_test

def get_model_transforms(net_type, dataset, num_classes):
    pre_trained_net = f'/kaggle/input/resnet-pth/{net_type}_{dataset}.pth'

    if net_type == 'densenet':
        if dataset == 'svhn':
            model = models.DenseNet3(100, num_classes)
            model.load_state_dict(torch.load(pre_trained_net, map_location = "cuda:" + str(0)))
        else:
            model = torch.load(pre_trained_net, map_location = "cuda:" + str(0))
        in_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((125.3/255, 123.0/255, 113.9/255), (63.0/255, 62.1/255.0, 66.7/255.0)),])
        
    elif net_type == 'resnet':
        model = models.ResNet34(num_c=num_classes)
        model.load_state_dict(torch.load(pre_trained_net, map_location = "cuda:" + str(0)))
        in_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),])
    else:
        raise ValueError()

    model.cuda()
    model.eval()

    return model, in_transform
    
def extract_activations(X, model, layer_idx, keep_grad=False, return_pred=False):
    if keep_grad:
        activations = model.intermediate_forward(X, layer_idx)
    else:
        with torch.no_grad():
            activations = model.intermediate_forward(X, layer_idx)

    activations = activations.view(activations.size(0), activations.size(1), -1)
    activations = torch.mean(activations, 2)

    if return_pred:
        output = model(X)
        y_pred = output.data.max(1)[1]
        return activations, y_pred
        
    else:
        return activations
        
def activations_from_loader(model, layer, loader):

    activations = []
    labels = []

    for x, y in loader:
        acts = extract_activations(x.cuda(), model, layer)

        activations.append(acts.cpu().numpy())
        labels.append(y.cpu().numpy())
    return np.concatenate(activations), np.concatenate(labels)

class Datasets():
    def __init__(self, ds_name, in_transform, net_type, adv_type, outf):
        self.ds_name = ds_name
        self.net_type = net_type
        self.adv_type = adv_type

        self.train_loader, _ = data_loader.getTargetDataSet(ds_name, 100, in_transform, './data')

        fname = f"{net_type}_{ds_name}_{adv_type}.pth"

        test_data = torch.load(f'{outf}/clean_data_{fname}')
        new_size = 100 * (len(test_data) // 100)
        test_data = test_data[:new_size]

        noisy_data = torch.load(f'{outf}/noisy_data_{fname}')[:new_size]
        adv_data = torch.load(f'{outf}/adv_data_{fname}')[:new_size]
        targets = torch.load(f'{outf}/label_{fname}').cpu().numpy()[:new_size]

        self.X_test = torch.cat([adv_data, test_data, noisy_data]).cuda()
        self.y_test = np.tile(targets, 3)
        self.adv_test = np.array([0] * len(adv_data) + [1] * (len(test_data) + len(noisy_data)))

        # Splits
        p_size = len(test_data)
        p_split = int(p_size*0.1)
        idxs_trainval = np.concatenate([
            np.arange(p_split),
            np.arange(p_size, p_size+p_split),
            np.arange(2*p_size, 2*p_size+p_split)
        ])

        self.idxs_test = np.delete(np.arange(len(self.X_test)), idxs_trainval)

        pivot = int(len(idxs_trainval) / 6)
        self.idxs_train = np.concatenate([idxs_trainval[:pivot], idxs_trainval[2*pivot:3*pivot], idxs_trainval[4*pivot:5*pivot]])
        self.idxs_val = np.concatenate([idxs_trainval[pivot:2*pivot], idxs_trainval[3*pivot:4*pivot], idxs_trainval[5*pivot:]])

class TrainValLoader:
    def __init__(self, model, ds, batch_size=100):
        self.ds = ds
        self.model = model
        self.batch_size = batch_size
        
    def __call__(self, layer_idx):
        X_train, y_train = activations_from_loader(self.model, layer_idx, self.ds.train_loader)
        adv_train = np.repeat(1, len(X_train))

        X_valid = self.ds.X_test[self.ds.idxs_val]
        acts_valid, y_valid = [], []

        for batch in torch.split(X_valid, self.batch_size):
            a_temp, y_temp = extract_activations(batch, self.model, layer_idx, return_pred=True)
            acts_valid.append(a_temp.cpu().numpy())
            y_valid.append(y_temp.cpu().numpy())

        X_valid = np.concatenate(acts_valid)
        y_valid = np.concatenate(y_valid)
        adv_valid = self.ds.adv_test[self.ds.idxs_val]

        return X_train, X_valid, y_train, y_valid, adv_train, adv_valid

class LabelledTrainLoader:
    def __init__(self, model, ds, batch_size=100):
        self.ds = ds
        self.model = model
        self.batch_size = batch_size
        
    def __call__(self, layer_idx):
        X_test = self.ds.X_test[self.ds.idxs_train]

        acts, y_test = [], []
        for batch in torch.split(X_test, self.batch_size):
            a_temp, y_temp = extract_activations(batch, self.model, layer_idx, return_pred=True)
            acts.append(a_temp.cpu().numpy())
            y_test.append(y_temp.cpu().numpy())
        acts = np.concatenate(acts)
        y_test = np.concatenate(y_test)

        return acts, y_test

class LabelledValLoader:
    def __init__(self, model, ds, batch_size=100):
        self.ds = ds
        self.model = model
        self.batch_size = batch_size
        
    def __call__(self, layer_idx):
        X_test = self.ds.X_test[self.ds.idxs_val]

        acts, y_test = [], []
        for batch in torch.split(X_test, self.batch_size):
            a_temp, y_temp = extract_activations(batch, self.model, layer_idx, return_pred=True)
            acts.append(a_temp.cpu().numpy())
            y_test.append(y_temp.cpu().numpy())
        acts = np.concatenate(acts)
        y_test = np.concatenate(y_test)

        return acts, y_test

class LabelledTestLoader:
    def __init__(self, model, ds, batch_size=100):
        self.ds = ds
        self.model = model
        self.batch_size = batch_size
        
    def __call__(self, layer_idx):
        X_test = self.ds.X_test[self.ds.idxs_test]

        acts, y_test = [], []
        for batch in torch.split(X_test, self.batch_size):
            a_temp, y_temp = extract_activations(batch, self.model, layer_idx, return_pred=True)
            acts.append(a_temp.cpu().numpy())
            y_test.append(y_temp.cpu().numpy())
        acts = np.concatenate(acts)
        y_test = np.concatenate(y_test)

        return acts, y_test

def idxs_train_val_test(data_size):
    p_size = data_size // 3
    p_split = int(p_size*0.1)
    idxs_trainval = np.concatenate([np.arange(p_split), np.arange(p_size, p_size+p_split),
        np.arange(2*p_size, 2*p_size+p_split)])

    idxs_test = np.delete(np.arange(data_size), idxs_trainval)

    pivot = int(len(idxs_trainval) / 6)
    idxs_train = np.concatenate([idxs_trainval[:pivot], idxs_trainval[2*pivot:3*pivot], idxs_trainval[4*pivot:5*pivot]])
    idxs_val = np.concatenate([idxs_trainval[pivot:2*pivot], idxs_trainval[3*pivot:4*pivot], idxs_trainval[5*pivot:]])

    return idxs_train, idxs_val, idxs_test

def idxs_train_val_test_ga(data_size):
    """Updated to match DatasetsGA 4-way split structure"""
    p_size = data_size // 3  # Still 3 parts: adv, clean, noisy
    
    # Use same split logic as DatasetsGA
    trainval_frac = 0.1
    ga_frac = 0.1
    
    train_val_span = int(p_size * trainval_frac)
    ga_span = int(p_size * ga_frac)
    
    # Training + Validation indices (10% total)
    idxs_trainval = np.concatenate([
        np.arange(train_val_span),
        np.arange(p_size, p_size + train_val_span),
        np.arange(2*p_size, 2*p_size + train_val_span)
    ])
    
    # GA Validation indices (10% total) 
    idxs_ga_val = np.concatenate([
        np.arange(train_val_span, train_val_span + ga_span),
        np.arange(p_size + train_val_span, p_size + train_val_span + ga_span),
        np.arange(2*p_size + train_val_span, 2*p_size + train_val_span + ga_span)
    ])
    
    # Test indices (remaining 80%)
    all_used_idxs = np.concatenate([idxs_trainval, idxs_ga_val])
    idxs_test = np.delete(np.arange(p_size * 3), all_used_idxs)
    
    # Split trainval into train and validation
    pivot = int(len(idxs_trainval) / 6)
    idxs_train = np.concatenate([
        idxs_trainval[:pivot],
        idxs_trainval[2*pivot:3*pivot],
        idxs_trainval[4*pivot:5*pivot]
    ])
    idxs_val = np.concatenate([
        idxs_trainval[pivot:2*pivot],
        idxs_trainval[3*pivot:4*pivot],
        idxs_trainval[5*pivot:]
    ])
    
    # Return 4 values to match DatasetsGA
    return idxs_train, idxs_val, idxs_ga_val, idxs_test

class LabelledGAValLoader:
    def __init__(self, model, ds, batch_size=100):
        self.ds = ds
        self.model = model
        self.batch_size = batch_size
        
    def __call__(self, layer_idx):
        # Fix: Use the GA validation indices to slice X_test, not use idxs_ga_val directly
        X_test = self.ds.X_test[self.ds.idxs_ga_val]  # This gives us the actual data

        acts, y_test = [], []
        for batch in torch.split(X_test, self.batch_size):  # Now split the tensor data
            a_temp, y_temp = extract_activations(batch, self.model, layer_idx, return_pred=True)
            acts.append(a_temp.cpu().numpy())
            y_test.append(y_temp.cpu().numpy())
        acts = np.concatenate(acts)
        y_test = np.concatenate(y_test)

        return acts, y_test

class DatasetsGA(Datasets):
    def __init__(self, ds_name, in_transform, net_type, adv_type, outf):
        # Call parent constructor to set up all basic data loading
        super().__init__(ds_name, in_transform, net_type, adv_type, outf)
        
        # Override splits with 4-way GA split
        p_size = len(self.X_test) // 3
        self.idxs_train, self.idxs_val, self.idxs_ga_val, self.idxs_test = self._train_val_ga_test_split(p_size)
        
    def _train_val_ga_test_split(self, split_size, trainval_frac=0.1, ga_frac=0.1):
        # 10% for training+validation (same as original)
        train_val_span = int(split_size * trainval_frac)
        
        # 10% for GA validation
        ga_span = int(split_size * ga_frac)
        
        # Training + Validation indices (10% total)
        idxs_trainval = np.concatenate([
            np.arange(train_val_span),
            np.arange(split_size, split_size + train_val_span),
            np.arange(2*split_size, 2*split_size + train_val_span)
        ])
        
        # GA Validation indices (10% total)
        idxs_ga_val = np.concatenate([
            np.arange(train_val_span, train_val_span + ga_span),
            np.arange(split_size + train_val_span, split_size + train_val_span + ga_span),
            np.arange(2*split_size + train_val_span, 2*split_size + train_val_span + ga_span)
        ])
        
        # Test indices (remaining 80%)
        all_used_idxs = np.concatenate([idxs_trainval, idxs_ga_val])
        idxs_test = np.delete(np.arange(split_size * 3), all_used_idxs)
        
        # Split trainval into train and validation (same as original)
        pivot = int(len(idxs_trainval) / 6)
        idxs_train = np.concatenate([
            idxs_trainval[:pivot],
            idxs_trainval[2*pivot:3*pivot],
            idxs_trainval[4*pivot:5*pivot]
        ])
        idxs_val = np.concatenate([
            idxs_trainval[pivot:2*pivot],
            idxs_trainval[3*pivot:4*pivot],
            idxs_trainval[5*pivot:]
        ])
        
        return idxs_train, idxs_val, idxs_ga_val, idxs_test

class GroupedScaler(BaseEstimator, TransformerMixin):
    def __init__(self, with_centering=True):
        self.with_centering = with_centering

    def fit(self, X, y=None):
        groups = X[:, -1].astype(int)
        X = X[:, :-1]

        if self.with_centering:
            self.group_means = []
            for lab in np.unique(groups):
                mask_idxs = (groups == lab).nonzero()[0]
                X_mean = np.mean(X[mask_idxs], axis=0)
                self.group_means.append(X_mean)
                
        return self
        
    def transform(self, X, y=None):
        groups = X[:, -1].astype(int)
        X = X[:, :-1]

        if self.with_centering:
            old_idxs = []
            X_norm = []
            for lab in np.unique(groups):

                mask_idxs = (groups == lab).nonzero()[0]
                temp_X = X[mask_idxs] - self.group_means[lab]

                X_norm.extend(temp_X)
                old_idxs.extend(mask_idxs)

            X = np.array(X_norm)[np.argsort(old_idxs)]

        return X
        
    def get_params(self, deep=True):
        return {'with_centering': self.with_centering}
    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

class Mahalanobis(BaseEstimator):
    def __init__(self, with_centering=True, centering="closest"):
        self.with_centering = with_centering
        self.centering = centering
    def fit(self, X, y=None):
        
        self.scaler = GroupedScaler(self.with_centering)
        X = self.scaler.fit_transform(X)

        self.detector_ = EmpiricalCovariance(store_precision=True, assume_centered=False)
        self.detector_.fit(X)

        return self
    def decision_function(self, X, y=None):
        
        if self.centering == 'closest':
            X = X[:,:-1]
            scores = []
            
            for mean in self.scaler.group_means:
                score = -0.5*np.diag(np.matmul(np.matmul((X-mean), self.detector_.precision_), (X-mean).T))
                scores.append(score)

            scores = np.array(scores)
            scores = np.max(scores, axis=0)
            
        elif self.centering == 'group':
            X = self.scaler.transform(X)
            scores = -0.5*np.diag(np.matmul(np.matmul(X, self.detector_.precision_), X.T))
        
        return scores

    def get_params(self, deep=True):
        return {'with_centering': self.with_centering}
        
    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

class NetDetector:
    def __init__(self, n_layers, detector_class, tqdm=True, logger=None, exp_name="",
        outf="", pre_computed=False, pre_computed_path="/kaggle/input/ocsvm-best-params/OCSVM_best_params.json"):
        
        self.detector_class = detector_class
        self.tqdm = not tqdm
        self.n_layers = n_layers
        self.logger = logger
        self.exp_name = exp_name
        self.outf = outf
        self.pre_computed = pre_computed
        self.pre_computed_path = pre_computed_path

    def fit(self, dl_train, dl_unseen_train, adv_unseen_train):
        self.logger.info(f"{self.exp_name}: training layer detectors...")
        self.detectors = self.train_layer_detectors(dl_train)
        
        train_scores = self.get_layer_scores(dl_unseen_train)
        self.logger.info(f"{self.exp_name}: training final logistic...")

        self.lr = self.train_logistic_regression(train_scores, adv_unseen_train)

    def predict(self, data_loader):
        
        predicted_scores = self.get_layer_scores(data_loader)
        metrics = self.get_final_metrics(predicted_scores)
        return predicted_scores, metrics
        
    def train_layer_detectors(self, data_loader):
        detectors = []
        for layer_idx in tqdm_auto(range(self.n_layers), disable=not self.tqdm, desc="Training layer detectors...",):
            self.logger.info(f"{self.exp_name}: started layer {layer_idx}")
            X_train, X_valid, y_train, y_valid, adv_train, adv_valid = data_loader(layer_idx)
            
            X = np.concatenate((X_train, X_valid))
            y = np.concatenate((y_train, y_valid))
            adv = np.concatenate((adv_train, adv_valid))

            # Debug: Print unique classes and counts in adv
            unique, counts = np.unique(adv, return_counts=True)
            print(f"Layer {layer_idx} adv label distribution: {dict(zip(unique, counts))}")

            if self.pre_computed:
                with open(self.pre_computed_path, "r") as fname:
                    params = json.load(fname)
                detector = clone(self.detector_class)

                best_params = params[f"{self.exp_name}_{layer_idx}"]
                detector = detector.set_params(**best_params).fit(np.c_[X_train, y_train])
                
            else:
                srch = clone(self.detector_class)
                srch.fit(np.c_[X, y], adv, callback=DeltaYStopper(0.02, n_best=10))

                with open(f"{self.outf}/bayes_{self.exp_name}_{layer_idx}.pkl", "wb") as outp:
                    pickle.dump(srch, outp, pickle.HIGHEST_PROTOCOL)

                detector = srch.estimator.set_params(**srch.best_params_).fit(np.c_[X_train, y_train])
                self.logger.info(f"{self.exp_name}: ACC = {srch.best_score_}, BEST_PARAMS = {srch.best_params_}")
                
            detectors.append(detector)

        return detectors

    def get_layer_scores(self, data_loader):
        scores = []
        for layer_idx in tqdm_auto(range(self.n_layers), disable=not self.tqdm, desc="Extracting layer scores...", leave=False):
            X, y = data_loader(layer_idx)
            scores.append(self.detectors[layer_idx].decision_function(np.c_[X, y]))
        
        return np.vstack(scores).T

    def train_logistic_regression(self, X, adv):
        lr = LogisticRegressionCV(penalty="l1", solver="liblinear", max_iter=10000, n_jobs=-1)
        lr.fit(X, adv)
        return lr
        
    def get_final_metrics(self, X):
        preds = self.lr.predict(X)
        probas = self.lr.predict_proba(X)

        return np.array([preds, probas[:, 0]]).T

def aupr(y_true, y_pred, pos_label=1):
    precision, recall, _ = precision_recall_curve(y_true, y_pred, pos_label=pos_label)
    return auc(recall, precision)

def get_curve(dir_name, stypes = ['Baseline', 'Gaussian_LDA']):
    tp, fp = dict(), dict()
    tnr_at_tpr95 = dict()
    for stype in stypes:
        known = np.loadtxt('{}/confidence_{}_In.txt'.format(dir_name, stype), delimiter='\n')
        novel = np.loadtxt('{}/confidence_{}_Out.txt'.format(dir_name, stype), delimiter='\n')
        known.sort()
        novel.sort()
        end = np.max([np.max(known), np.max(novel)])
        start = np.min([np.min(known),np.min(novel)])
        num_k = known.shape[0]
        num_n = novel.shape[0]
        tp[stype] = -np.ones([num_k+num_n+1], dtype=int)
        fp[stype] = -np.ones([num_k+num_n+1], dtype=int)
        tp[stype][0], fp[stype][0] = num_k, num_n
        k, n = 0, 0
        for l in range(num_k+num_n):
            if k == num_k:
                tp[stype][l+1:] = tp[stype][l]
                fp[stype][l+1:] = np.arange(fp[stype][l]-1, -1, -1)
                break
            elif n == num_n:
                tp[stype][l+1:] = np.arange(tp[stype][l]-1, -1, -1)
                fp[stype][l+1:] = fp[stype][l]
                break
            else:
                if novel[n] < known[k]:
                    n += 1
                    tp[stype][l+1] = tp[stype][l]
                    fp[stype][l+1] = fp[stype][l] - 1
                else:
                    k += 1
                    tp[stype][l+1] = tp[stype][l] - 1
                    fp[stype][l+1] = fp[stype][l]
        tpr95_pos = np.abs(tp[stype] / num_k - .95).argmin()
        tnr_at_tpr95[stype] = 1. - fp[stype][tpr95_pos] / num_n
    return tp, fp, tnr_at_tpr95
    
def metric(dir_name, stypes = ['Bas', 'Gau'], verbose=False):
    tp, fp, tnr_at_tpr95 = get_curve(dir_name, stypes)
    results = dict()
    mtypes = ['TNR', 'AUROC', 'DTACC', 'AUIN', 'AUOUT']
    if verbose:
        print('      ', end='')
        for mtype in mtypes:
            print(' {mtype:6s}'.format(mtype=mtype), end='')
        print('')
        
    for stype in stypes:
        if verbose:
            print('{stype:5s} '.format(stype=stype), end='')
        results[stype] = dict()
        
        # TNR
        mtype = 'TNR'
        results[stype][mtype] = tnr_at_tpr95[stype]
        if verbose:
            print(' {val:6.3f}'.format(val=100.*results[stype][mtype]), end='')
        
        # AUROC
        mtype = 'AUROC'
        tpr = np.concatenate([[1.], tp[stype]/tp[stype][0], [0.]])
        fpr = np.concatenate([[1.], fp[stype]/fp[stype][0], [0.]])
        results[stype][mtype] = -np.trapz(1.-fpr, tpr)
        if verbose:
            print(' {val:6.3f}'.format(val=100.*results[stype][mtype]), end='')
        
        # DTACC
        mtype = 'DTACC'
        results[stype][mtype] = .5 * (tp[stype]/tp[stype][0] + 1.-fp[stype]/fp[stype][0]).max()
        if verbose:
            print(' {val:6.3f}'.format(val=100.*results[stype][mtype]), end='')
        
        # AUIN
        mtype = 'AUIN'
        denom = tp[stype]+fp[stype]
        denom[denom == 0.] = -1.
        pin_ind = np.concatenate([[True], denom > 0., [True]])
        pin = np.concatenate([[.5], tp[stype]/denom, [0.]])
        results[stype][mtype] = -np.trapz(pin[pin_ind], tpr[pin_ind])
        if verbose:
            print(' {val:6.3f}'.format(val=100.*results[stype][mtype]), end='')
        
        # AUOUT
        mtype = 'AUOUT'
        denom = tp[stype][0]-tp[stype]+fp[stype][0]-fp[stype]
        denom[denom == 0.] = -1.
        pout_ind = np.concatenate([[True], denom > 0., [True]])
        pout = np.concatenate([[0.], (fp[stype][0]-fp[stype])/denom, [.5]])
        results[stype][mtype] = np.trapz(pout[pout_ind], 1.-fpr[pout_ind])
        if verbose:
            print(' {val:6.3f}'.format(val=100.*results[stype][mtype]), end='')
            print('')
    
    return results

def mle_batch(data, batch, k):
    data = np.asarray(data, dtype=np.float32)
    batch = np.asarray(batch, dtype=np.float32)

    k = min(k, len(data)-1)
    f = lambda v: - k / np.sum(np.log(v/v[-1]))
    a = cdist(batch, data)
    a = np.apply_along_axis(np.sort, axis=1, arr=a)[:,1:k+1]
    a = np.apply_along_axis(f, axis=1, arr=a)
    return a
    
def merge_and_generate_labels(X_pos, X_neg):
    """
    merge positive and negative artifact and generate labels
    X_pos: adversarial samples -> label 0
    X_neg: clean/noisy samples -> label 1
    """
    X_pos = np.asarray(X_pos, dtype=np.float32)
    X_pos = X_pos.reshape((X_pos.shape[0], -1))

    X_neg = np.asarray(X_neg, dtype=np.float32)
    X_neg = X_neg.reshape((X_neg.shape[0], -1))

    X = np.concatenate((X_pos, X_neg))
    # Changed: adversarial -> 0, clean -> 1
    y = np.concatenate((np.zeros(X_pos.shape[0]), np.ones(X_neg.shape[0])))
    y = y.reshape((X.shape[0], 1))

    return X, y
    
def sample_estimator(model, num_classes, feature_list, train_loader):
    """
    compute sample mean and precision (inverse of covariance)
    return: sample_class_mean: list of class mean
             precision: list of precisions
    """
    import sklearn.covariance
    
    model.eval()
    group_lasso = sklearn.covariance.EmpiricalCovariance(assume_centered=False)
    correct, total = 0, 0
    num_output = len(feature_list)
    num_sample_per_class = np.empty(num_classes)
    num_sample_per_class.fill(0)
    list_features = []
    
    for i in range(num_output):
        temp_list = []
        for j in range(num_classes):
            temp_list.append(0)
        list_features.append(temp_list)
    
    for data, target in train_loader:
        total += data.size(0)
        data = data.cuda()
        data = Variable(data, volatile=True)
        output, out_features = model.feature_list(data)
        
        # get hidden features
        for i in range(num_output):
            out_features[i] = out_features[i].view(out_features[i].size(0), out_features[i].size(1), -1)
            out_features[i] = torch.mean(out_features[i].data, 2)
            
        # compute the accuracy
        pred = output.data.max(1)[1]
        equal_flag = pred.eq(target.cuda()).cpu()
        correct += equal_flag.sum()
        
        # construct the sample matrix
        for i in range(data.size(0)):
            label = target[i]
            if num_sample_per_class[label] == 0:
                out_count = 0
                for out in out_features:
                    list_features[out_count][label] = out[i].view(1, -1)
                    out_count += 1
            else:
                out_count = 0
                for out in out_features:
                    list_features[out_count][label] \
                    = torch.cat((list_features[out_count][label], out[i].view(1, -1)), 0)
                    out_count += 1                
            num_sample_per_class[label] += 1
            
    sample_class_mean = []
    out_count = 0
    for num_feature in feature_list:
        temp_list = torch.Tensor(num_classes, int(num_feature)).cuda()
        for j in range(num_classes):
            temp_list[j] = torch.mean(list_features[out_count][j], 0)
        sample_class_mean.append(temp_list)
        out_count += 1
        
    precision = []
    for k in range(num_output):
        X = 0
        for i in range(num_classes):
            if i == 0:
                X = list_features[k][i] - sample_class_mean[k][i]
            else:
                X = torch.cat((X, list_features[k][i] - sample_class_mean[k][i]), 0)
                
        # find inverse            
        group_lasso.fit(X.cpu().numpy())
        temp_precision = group_lasso.precision_
        temp_precision = torch.from_numpy(temp_precision).float().cuda()
        precision.append(temp_precision)
        
    print('\n Training Accuracy:({:.2f}%)\n'.format(100. * correct / total))

    return sample_class_mean, precision
    
def get_Mahalanobis_score(model, test_loader, num_classes, outf, out_flag, net_type, sample_mean, precision, layer_index, magnitude):
    model.eval()
    Mahalanobis = []
    
    if out_flag == True:
        temp_file_name = '%s/confidence_Ga%s_In.txt'%(outf, str(layer_index))
    else:
        temp_file_name = '%s/confidence_Ga%s_Out.txt'%(outf, str(layer_index))
        
    g = open(temp_file_name, 'w')
    
    for data, target in test_loader:
        
        data, target = data.cuda(), target.cuda()
        data, target = Variable(data, requires_grad = True), Variable(target)
        
        out_features = model.intermediate_forward(data, layer_index)
        out_features = out_features.view(out_features.size(0), out_features.size(1), -1)
        out_features = torch.mean(out_features, 2)
        
        # compute Mahalanobis score
        gaussian_score = 0
        for i in range(num_classes):
            batch_sample_mean = sample_mean[layer_index][i]
            zero_f = out_features.data - batch_sample_mean
            term_gau = -0.5*torch.mm(torch.mm(zero_f, precision[layer_index]), zero_f.t()).diag()
            if i == 0:
                gaussian_score = term_gau.view(-1,1)
            else:
                gaussian_score = torch.cat((gaussian_score, term_gau.view(-1,1)), 1)
        
        # Input_processing
        sample_pred = gaussian_score.max(1)[1]
        batch_sample_mean = sample_mean[layer_index].index_select(0, sample_pred)
        zero_f = out_features - Variable(batch_sample_mean)
        pure_gau = -0.5*torch.mm(torch.mm(zero_f, Variable(precision[layer_index])), zero_f.t()).diag()
        loss = torch.mean(-pure_gau)
        loss.backward()
         
        gradient =  torch.ge(data.grad.data, 0)
        gradient = (gradient.float() - 0.5) * 2
        if net_type == 'densenet':
            gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (63.0/255.0))
            gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (62.1/255.0))
            gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (66.7/255.0))
        elif net_type == 'resnet':
            gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (0.2023))
            gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (0.1994))
            gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (0.2010))
        tempInputs = torch.add(data.data, -magnitude, gradient)
 
        noise_out_features = model.intermediate_forward(Variable(tempInputs, volatile=True), layer_index)
        noise_out_features = noise_out_features.view(noise_out_features.size(0), noise_out_features.size(1), -1)
        noise_out_features = torch.mean(noise_out_features, 2)
        noise_gaussian_score = 0
        for i in range(num_classes):
            batch_sample_mean = sample_mean[layer_index][i]
            zero_f = noise_out_features.data - batch_sample_mean
            term_gau = -0.5*torch.mm(torch.mm(zero_f, precision[layer_index]), zero_f.t()).diag()
            if i == 0:
                noise_gaussian_score = term_gau.view(-1,1)
            else:
                noise_gaussian_score = torch.cat((noise_gaussian_score, term_gau.view(-1,1)), 1)      

        noise_gaussian_score, _ = torch.max(noise_gaussian_score, dim=1)
        Mahalanobis.extend(noise_gaussian_score.cpu().numpy())
        
        for i in range(data.size(0)):
            g.write("{}\n".format(noise_gaussian_score[i]))
    g.close()

    return Mahalanobis
def get_posterior(model, net_type, test_loader, magnitude, temperature, outf, out_flag):
    criterion = nn.CrossEntropyLoss()
    model.eval()
    total = 0
    if out_flag == True:
        temp_file_name_val = '%s/confidence_PoV_In.txt'%(outf)
        temp_file_name_test = '%s/confidence_PoT_In.txt'%(outf)
    else:
        temp_file_name_val = '%s/confidence_PoV_Out.txt'%(outf)
        temp_file_name_test = '%s/confidence_PoT_Out.txt'%(outf)
        
    g = open(temp_file_name_val, 'w')
    f = open(temp_file_name_test, 'w')
    
    for data, _ in test_loader:
        total += data.size(0)
        data = data.cuda()
        data = Variable(data, requires_grad = True)
        batch_output = model(data)
            
        # temperature scaling
        outputs = batch_output / temperature
        labels = outputs.data.max(1)[1]
        labels = Variable(labels)
        loss = criterion(outputs, labels)
        loss.backward()
         
        # Normalizing the gradient to binary in {0, 1}
        gradient =  torch.ge(data.grad.data, 0)
        gradient = (gradient.float() - 0.5) * 2
        if net_type == 'densenet':
            gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (63.0/255.0))
            gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (62.1/255.0))
            gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (66.7/255.0))
        elif net_type == 'resnet':
            gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (0.2023))
            gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (0.1994))
            gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (0.2010))

        tempInputs = torch.add(data.data,  -magnitude, gradient)
        outputs = model(Variable(tempInputs, volatile=True))
        outputs = outputs / temperature
        soft_out = F.softmax(outputs, dim=1)
        soft_out, _ = torch.max(soft_out.data, dim=1)
        
        for i in range(data.size(0)):
            if total <= 1000:
                g.write("{}\n".format(soft_out[i]))
            else:
                f.write("{}\n".format(soft_out[i]))
                
    f.close()
    g.close()
    
def get_Mahalanobis_score_adv(model, test_data, test_label, num_classes, outf, net_type, sample_mean, precision, layer_index, magnitude):
    model.eval()
    Mahalanobis = []
    batch_size = 100
    total = 0
    
    for data_index in range(int(np.floor(test_data.size(0)/batch_size))):
        target = test_label[total : total + batch_size].cuda()
        data = test_data[total : total + batch_size].cuda()
        total += batch_size
        data, target = Variable(data, requires_grad = True), Variable(target)
        
        out_features = model.intermediate_forward(data, layer_index)
        out_features = out_features.view(out_features.size(0), out_features.size(1), -1)
        out_features = torch.mean(out_features, 2)
        
        gaussian_score = 0
        for i in range(num_classes):
            batch_sample_mean = sample_mean[layer_index][i]
            zero_f = out_features.data - batch_sample_mean
            term_gau = -0.5*torch.mm(torch.mm(zero_f, precision[layer_index]), zero_f.t()).diag()
            if i == 0:
                gaussian_score = term_gau.view(-1,1)
            else:
                gaussian_score = torch.cat((gaussian_score, term_gau.view(-1,1)), 1)
        
        # Input_processing
        sample_pred = gaussian_score.max(1)[1]
        batch_sample_mean = sample_mean[layer_index].index_select(0, sample_pred)
        zero_f = out_features - Variable(batch_sample_mean)
        pure_gau = -0.5*torch.mm(torch.mm(zero_f, Variable(precision[layer_index])), zero_f.t()).diag()
        loss = torch.mean(-pure_gau)
        loss.backward()
         
        gradient =  torch.ge(data.grad.data, 0)
        gradient = (gradient.float() - 0.5) * 2
        if net_type == 'densenet':
            gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (63.0/255.0))
            gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (62.1/255.0))
            gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (66.7/255.0))
        elif net_type == 'resnet':
            gradient.index_copy_(1, torch.LongTensor([0]).cuda(), gradient.index_select(1, torch.LongTensor([0]).cuda()) / (0.2023))
            gradient.index_copy_(1, torch.LongTensor([1]).cuda(), gradient.index_select(1, torch.LongTensor([1]).cuda()) / (0.1994))
            gradient.index_copy_(1, torch.LongTensor([2]).cuda(), gradient.index_select(1, torch.LongTensor([2]).cuda()) / (0.2010))
        tempInputs = torch.add(data.data, -magnitude, gradient)
 
        noise_out_features = model.intermediate_forward(Variable(tempInputs, volatile=True), layer_index)
        noise_out_features = noise_out_features.view(noise_out_features.size(0), noise_out_features.size(1), -1)
        noise_out_features = torch.mean(noise_out_features, 2)
        noise_gaussian_score = 0
        for i in range(num_classes):
            batch_sample_mean = sample_mean[layer_index][i]
            zero_f = noise_out_features.data - batch_sample_mean
            term_gau = -0.5*torch.mm(torch.mm(zero_f, precision[layer_index]), zero_f.t()).diag()
            if i == 0:
                noise_gaussian_score = term_gau.view(-1,1)
            else:
                noise_gaussian_score = torch.cat((noise_gaussian_score, term_gau.view(-1,1)), 1)      

        noise_gaussian_score, _ = torch.max(noise_gaussian_score, dim=1)
        Mahalanobis.extend(noise_gaussian_score.cpu().numpy())
        
    return Mahalanobis
    
def get_LID(model, test_clean_data, test_adv_data, test_noisy_data, test_label, num_output):
    model.eval()  
    total = 0
    batch_size = 100
    
    LID, LID_adv, LID_noisy = [], [], []    
    overlap_list = [10, 20, 30, 40, 50, 60, 70, 80, 90]
    for i in overlap_list:
        LID.append([])
        LID_adv.append([])
        LID_noisy.append([])
        
    for data_index in range(int(np.floor(test_clean_data.size(0)/batch_size))):
        data = test_clean_data[total : total + batch_size].cuda()
        adv_data = test_adv_data[total : total + batch_size].cuda()
        noisy_data = test_noisy_data[total : total + batch_size].cuda()
        target = test_label[total : total + batch_size].cuda()

        total += batch_size
        
        # Use torch.no_grad() instead of volatile
        with torch.no_grad():
            # Process clean data
            output, out_features = model.feature_list(data)
            X_act = []
            for i in range(num_output):
                out_features[i] = out_features[i].view(out_features[i].size(0), out_features[i].size(1), -1)
                out_features[i] = torch.mean(out_features[i], 2)
                # Move tensor to CPU before NumPy conversion
                X_act.append(out_features[i].cpu().numpy().reshape((out_features[i].size(0), -1)))
            
            # Process adversarial data
            output, out_features = model.feature_list(adv_data)
            X_act_adv = []
            for i in range(num_output):
                out_features[i] = out_features[i].view(out_features[i].size(0), out_features[i].size(1), -1)
                out_features[i] = torch.mean(out_features[i], 2)
                # Move tensor to CPU before NumPy conversion
                X_act_adv.append(out_features[i].cpu().numpy().reshape((out_features[i].size(0), -1)))

            # Process noisy data
            output, out_features = model.feature_list(noisy_data)
            X_act_noisy = []
            for i in range(num_output):
                out_features[i] = out_features[i].view(out_features[i].size(0), out_features[i].size(1), -1)
                out_features[i] = torch.mean(out_features[i], 2)
                # Move tensor to CPU before NumPy conversion
                X_act_noisy.append(out_features[i].cpu().numpy().reshape((out_features[i].size(0), -1)))
        
        # LID
        list_counter = 0 
        for overlap in overlap_list:
            LID_list = []
            LID_adv_list = []
            LID_noisy_list = []

            for j in range(num_output):
                lid_score = mle_batch(X_act[j], X_act[j], k = overlap)
                lid_score = lid_score.reshape((lid_score.shape[0], -1))
                lid_adv_score = mle_batch(X_act[j], X_act_adv[j], k = overlap)
                lid_adv_score = lid_adv_score.reshape((lid_adv_score.shape[0], -1))
                lid_noisy_score = mle_batch(X_act[j], X_act_noisy[j], k = overlap)
                lid_noisy_score = lid_noisy_score.reshape((lid_noisy_score.shape[0], -1))
                
                LID_list.append(lid_score)
                LID_adv_list.append(lid_adv_score)
                LID_noisy_list.append(lid_noisy_score)

            LID_concat = LID_list[0]
            LID_adv_concat = LID_adv_list[0]
            LID_noisy_concat = LID_noisy_list[0]

            for i in range(1, num_output):
                LID_concat = np.concatenate((LID_concat, LID_list[i]), axis=1)
                LID_adv_concat = np.concatenate((LID_adv_concat, LID_adv_list[i]), axis=1)
                LID_noisy_concat = np.concatenate((LID_noisy_concat, LID_noisy_list[i]), axis=1)
                
            LID[list_counter].extend(LID_concat)
            LID_adv[list_counter].extend(LID_adv_concat)
            LID_noisy[list_counter].extend(LID_noisy_concat)
            list_counter += 1
            
    return LID, LID_adv, LID_noisy

import pickle
from torch.autograd import Variable

def compute_lid_training_features(model, train_loader, num_output):
    """
    Compute training features for LID computation.
    Returns a list of feature arrays, one per layer.
    """
    features = [[] for _ in range(num_output)]
    with torch.no_grad():
        for batch_idx, (inputs, _) in enumerate(train_loader):
            inputs = inputs.cuda()
            _, layer_outputs = model.feature_list(inputs)
            for layer_idx, output in enumerate(layer_outputs):
                # Average spatially: [batch_size, channels, H, W] -> [batch_size, channels]
                output = output.view(output.size(0), output.size(1), -1).mean(dim=2)
                features[layer_idx].append(output.cpu().numpy())
    
    # Concatenate features per layer: list of [n_samples, channels]
    for layer_idx in range(num_output):
        features[layer_idx] = np.concatenate(features[layer_idx], axis=0)
        print(f"Layer {layer_idx} feature shape: {features[layer_idx].shape}")
        #Layer 0 feature shape: (50000, 64)
        #Layer 1 feature shape: (50000, 64)
        #Layer 2 feature shape: (50000, 128)
        #Layer 3 feature shape: (50000, 256)
        #Layer 4 feature shape: (50000, 512)
    
    return features  # List of [n_samples, channels] for each layer

def main_break(outf, net_type, dataset, adv_type, gpu, num_classes, batch_size, dataroot):
    # Set the path to pre-trained model and output
    pre_trained_net = f'/kaggle/input/resnet-pth/{net_type}_{dataset}.pth'
    outf = os.path.join(outf, f'{net_type}_{dataset}/{adv_type}')
    if not os.path.isdir(outf):
        os.makedirs(outf)
    torch.cuda.manual_seed(0)
    torch.cuda.set_device(gpu)
    # Check the in-distribution dataset
    if dataset == 'cifar100':
        num_classes = 100
        
    try:
        if net_type == 'densenet':
            if dataset == 'svhn':
                model = models.DenseNet3(100, int(num_classes))
                model.load_state_dict(torch.load(pre_trained_net, map_location=f"cuda:{gpu}"))
            else:
                model = torch.load(pre_trained_net, map_location=f"cuda:{gpu}")
            in_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((125.3/255, 123.0/255, 113.9/255), (63.0/255, 62.1/255.0, 66.7/255.0))
            ])
        elif net_type == 'resnet':
            model = models.ResNet34(num_c=num_classes)
            model.load_state_dict(torch.load(pre_trained_net, map_location=f"cuda:{gpu}"))
            in_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
            ])
        model.cuda()
        print(f'Loaded model: {net_type}')
    except Exception as e:
        print(f"Error loading model: {e}")
        return
    
    # Load dataset
    print(f'Loading target data: {dataset}')
    try:
        train_loader, _ = data_loader.getTargetDataSet(dataset, batch_size, in_transform, dataroot)
        data_path = f'/kaggle/input/attacked-pth-files/{dataset.upper()}/{adv_type}/'
        test_clean_data = torch.load(os.path.join(data_path, f'clean_data_{net_type}_{dataset}_{adv_type}.pth'))
        test_adv_data = torch.load(os.path.join(data_path, f'adv_data_{net_type}_{dataset}_{adv_type}.pth'))
        test_noisy_data = torch.load(os.path.join(data_path, f'noisy_data_{net_type}_{dataset}_{adv_type}.pth'))
        test_label = torch.load(os.path.join(data_path, f'label_{net_type}_{dataset}_{adv_type}.pth'))
    except FileNotFoundError as e:
        print(f"Error loading data files: {e}")
        return

    # Set information about feature extraction
    model.eval()
    try:
        temp_x = torch.rand(2, 3, 32, 32).cuda()
        temp_x = Variable(temp_x)
        temp_list = model.feature_list(temp_x)[1]
        num_output = len(temp_list)
        feature_list = np.empty(num_output)
        count = 0
        for out in temp_list:
            feature_list[count] = out.size(1)
            count += 1
        print(f'Number of output layers: {num_output}, Feature dimensions: {feature_list}')
    except Exception as e:
        print(f"Error setting feature extraction: {e}")
        return
        
    # Compute and save Mahalanobis statistics
    print('Computing sample mean and covariance')
    try:
        sample_mean, precision = sample_estimator(model, num_classes, feature_list, train_loader)
        mahalanobis_stats_path = os.path.join(outf, f'{dataset}_{adv_type}_mahalanobis_stats.pkl')
        with open(mahalanobis_stats_path, 'wb') as f:
            pickle.dump((sample_mean, precision), f)
        print(f'Saved Mahalanobis statistics to {mahalanobis_stats_path}')
    except Exception as e:
        print(f"Error computing Mahalanobis statistics: {e}")
        return
    
    # Compute and save LID training features
    print('Computing LID training features')
    try:
        lid_training_features = compute_lid_training_features(model, train_loader, num_output)
        lid_features_path = os.path.join(outf, f'{dataset}_{adv_type}_lid_training_features.pkl')
        with open(lid_features_path, 'wb') as f:
            pickle.dump(lid_training_features, f)
        print(f'Saved LID training features to {lid_features_path}')
    except Exception as e:
        print(f"Error computing LID training features: {e}")
        return

    print('Computing LID scores')
    try:
        LID, LID_adv, LID_noisy = get_LID(model, test_clean_data, test_adv_data, test_noisy_data, test_label, num_output)
        print(f"LID shapes: {np.shape(LID)}, {np.shape(LID_adv)}, {np.shape(LID_noisy)}")
        print("LID[:3]: ", LID[:3])
        print("LID_adv[:3]: ", LID_adv[:3])
        print("LID_noisy[:3]: ", LID_noisy[:3])
    except Exception as e:
        print(f"Error computing LID scores: {e}")
        return

    overlap_list = [10, 20, 30, 40, 50, 60, 70, 80, 90]
    list_counter = 0
    for overlap in overlap_list:
        try:
            Save_LID = np.asarray(LID[list_counter], dtype=np.float32)
            Save_LID_adv = np.asarray(LID_adv[list_counter], dtype=np.float32)
            Save_LID_noisy = np.asarray(LID_noisy[list_counter], dtype=np.float32)
            Save_LID_pos = np.concatenate((Save_LID, Save_LID_noisy))
            LID_data, LID_labels = merge_and_generate_labels(Save_LID_adv, Save_LID_pos)
            file_name = os.path.join(outf, f'LID_{overlap}_{dataset}_{adv_type}.npy')
            LID_data = np.concatenate((LID_data, LID_labels), axis=1)
            np.save(file_name, LID_data)
            print(f'Saved LID data to {file_name}')
        except Exception as e:
            print(f"Error saving LID data for overlap {overlap}: {e}")
        list_counter += 1
    
    print('Computing Mahalanobis scores')
    m_list = [0.0, 0.01, 0.005, 0.002, 0.0014, 0.001, 0.0005]
    for magnitude in m_list:
        print(f'\nNoise: {magnitude}')
        try:
            Mahalanobis_in = None
            Mahalanobis_out = None
            Mahalanobis_noisy = None
            for i in range(num_output):
                M_in = get_Mahalanobis_score_adv(model, test_clean_data, test_label, num_classes, outf, net_type, sample_mean, precision, i, magnitude)
                M_in = np.asarray(M_in, dtype=np.float32)
                if i == 0:
                    Mahalanobis_in = M_in.reshape((M_in.shape[0], -1))
                else:
                    Mahalanobis_in = np.concatenate((Mahalanobis_in, M_in.reshape((M_in.shape[0], -1))), axis=1)

                M_out = get_Mahalanobis_score_adv(model, test_adv_data, test_label, num_classes, outf, net_type, sample_mean, precision, i, magnitude)
                M_out = np.asarray(M_out, dtype=np.float32)
                if i == 0:
                    Mahalanobis_out = M_out.reshape((M_out.shape[0], -1))
                else:
                    Mahalanobis_out = np.concatenate((Mahalanobis_out, M_out.reshape((M_out.shape[0], -1))), axis=1)
                
                M_noisy = get_Mahalanobis_score_adv(model, test_noisy_data, test_label, num_classes, outf, net_type, sample_mean, precision, i, magnitude)
                M_noisy = np.asarray(M_noisy, dtype=np.float32)
                if i == 0:
                    Mahalanobis_noisy = M_noisy.reshape((M_noisy.shape[0], -1))
                else:
                    Mahalanobis_noisy = np.concatenate((Mahalanobis_noisy, M_noisy.reshape((M_noisy.shape[0], -1))), axis=1)
                
            Mahalanobis_in = np.asarray(Mahalanobis_in, dtype=np.float32)
            Mahalanobis_out = np.asarray(Mahalanobis_out, dtype=np.float32)
            Mahalanobis_noisy = np.asarray(Mahalanobis_noisy, dtype=np.float32)
            Mahalanobis_pos = np.concatenate((Mahalanobis_in, Mahalanobis_noisy))

            Mahalanobis_data, Mahalanobis_labels = merge_and_generate_labels(Mahalanobis_out, Mahalanobis_pos)
            file_name = os.path.join(outf, f'Mahalanobis_{magnitude}_{dataset}_{adv_type}.npy')
            Mahalanobis_data = np.concatenate((Mahalanobis_data, Mahalanobis_labels), axis=1)
            np.save(file_name, Mahalanobis_data)
            print(f'Saved Mahalanobis data to {file_name}')
        except Exception as e:
            print(f"Error processing Mahalanobis scores for magnitude {magnitude}: {e}")

def mainb(net_type, dataset, outf, gpu, num_classes, adv_type):
    # set the path to pre-trained model and output
    pre_trained_net = '/kaggle/input/resnet-pth/' + net_type + '_' + dataset + '.pth'
    outf = outf + net_type + '_' + dataset + '/'
    if not os.path.isdir(outf):
        os.makedirs(outf)
    torch.cuda.manual_seed(0)
    torch.cuda.set_device(gpu)

    # check the in-distribution dataset
    if dataset == 'cifar100':
        num_classes = 100
        
    # load networks
    if net_type == 'densenet':
        if dataset == 'svhn':
            model = models.DenseNet3(100, int(num_classes))
            model.load_state_dict(torch.load(pre_trained_net, map_location="cuda:" + str(args.gpu)))
        else:
            model = torch.load(pre_trained_net, map_location="cuda:" + str(gpu))
        in_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((125.3/255, 123.0/255, 113.9/255), (63.0/255, 62.1/255.0, 66.7/255.0)),
        ])
    elif net_type == 'resnet':
        model = models.ResNet34(num_c=num_classes)
        model.load_state_dict(torch.load(pre_trained_net, map_location="cuda:" + str(gpu)))
        in_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])
    model.cuda()
    print('load model: ' + net_type)
    
    # load dataset
    print('load target data: ', dataset)
    train_loader, _ = data_loader.getTargetDataSet(dataset, batch_size, in_transform, dataroot)
    test_clean_data = torch.load(f'/kaggle/input/attacked-pth-files/{dataset.upper()}/{adv_type}/' + 'clean_data_%s_%s_%s.pth' % (net_type, dataset, adv_type))
    test_adv_data = torch.load(f'/kaggle/input/attacked-pth-files/{dataset.upper()}/{adv_type}/' + 'adv_data_%s_%s_%s.pth' % (net_type, dataset, adv_type))
    test_noisy_data = torch.load(f'/kaggle/input/attacked-pth-files/{dataset.upper()}/{adv_type}/' + 'noisy_data_%s_%s_%s.pth' % (net_type, dataset, adv_type))
    test_label = torch.load(f'/kaggle/input/attacked-pth-files/{dataset.upper()}/{adv_type}/' + 'label_%s_%s_%s.pth' % (net_type, dataset, adv_type))

    # Lưu 10 ảnh clean, adv, và noisy
    num_samples = min(10, test_clean_data.size(0))  # Đảm bảo không vượt quá số mẫu có sẵn
    clean_samples = test_clean_data[:num_samples]
    adv_samples = test_adv_data[:num_samples]
    noisy_samples = test_noisy_data[:num_samples]

    # Đảo ngược chuẩn hóa
    if net_type == 'densenet':
        mean = torch.tensor([125.3/255, 123.0/255, 113.9/255]).view(1, 3, 1, 1)
        std = torch.tensor([63.0/255, 62.1/255.0, 66.7/255.0]).view(1, 3, 1, 1)
    else:  # resnet
        mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
        std = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)

    clean_samples = clean_samples * std + mean  # Đảo ngược chuẩn hóa
    adv_samples = adv_samples * std + mean
    noisy_samples = noisy_samples * std + mean

    # Đảm bảo giá trị pixel nằm trong [0, 1]
    clean_samples = torch.clamp(clean_samples, 0, 1)
    adv_samples = torch.clamp(adv_samples, 0, 1)
    noisy_samples = torch.clamp(noisy_samples, 0, 1)

    image_dir = '/kaggle/working/images'
    os.makedirs(image_dir, exist_ok=True)

    for i in range(num_samples):
        vutils.save_image(clean_samples[i], f'{image_dir}/{dataset}_{adv_type}_clean_image_{i}.png')
        vutils.save_image(adv_samples[i], f'{image_dir}/{dataset}_{adv_type}_adv_image_{i}.png')
        vutils.save_image(noisy_samples[i], f'{image_dir}/{dataset}_{adv_type}_noisy_image_{i}.png')

    print(f'Saved {num_samples} clean, adv, and noisy images to {image_dir}')

def run_maha(ds_name, net_type, adv_type, outf):
    outf = f"{outf}/{net_type}_{ds_name}/{adv_type}"
    
    for method in ['Mahalanobis', 'LID']:
        print(f'Started model selection for {method}...')
        regex = r'(\d+)' if method == 'LID' else r'(\d+\.\d+)'

        params = re.findall(f'{method}_{regex}_{ds_name}_{adv_type}', ''.join(os.listdir(outf)))
        params = np.array(params).astype(int if method == 'LID' else float)
        print(f'{method} params list: {", ".join(np.sort(params.astype(str)))}')

        aurocs = []

        for param in params:
            data = np.load(f'{outf}/{method}_{param}_{ds_name}_{adv_type}.npy')

            # Adv label
            adv =  data[:, -1]
            # Use coherent labelling, i.e. adv -> -1, not adv -> 1
            adv = np.select([adv==0, adv==1], [0, 1])

            # Load spltting idxs
            idxs_train, idxs_val, idxs_val_ga, idxs_test = idxs_train_val_test_ga(len(data))

            X_train = data[idxs_train][:,:n_layers(net_type)]
            adv_train = adv[idxs_train]

            lr = LogisticRegressionCV(penalty='l1', solver='liblinear', max_iter=10000, n_jobs=-1)
            lr.fit(X_train, adv_train)

            X_valid = data[idxs_val][:,:n_layers(net_type)]
            adv_conf_valid = lr.predict_proba(X_valid)[:, 0]
            auroc_valid = roc_auc_score(adv[idxs_val], -adv_conf_valid)
            aurocs.append(auroc_valid)

        aurocs = np.array(aurocs)
        idx_max = np.argmax(aurocs)

        print(f'{method} with param {params[idx_max]} has AUROC = {round(aurocs[idx_max]*100, 2)} on the validation set.')

        X = np.load(f'{outf}/{method}_{params[idx_max]}_{ds_name}_{adv_type}.npy')
        np.save(f'/kaggle/working/{method}_best_{ds_name}_{adv_type}.npy', X)

def run_ocsvm(net_type, ds_name, adv_type, outf='/kaggle/input/attacked-pth-files', ocsvm_fname='ocsvm', bayes_n_points=1, bayes_n_iter=10, use_logger=True, batch_size = 100, pre_computed=True):

    outf = outf + f"/{ds_name.upper()}/{adv_type}"
    data_outf = f"/kaggle/working/{ds_name.upper()}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)

    data_outf = f"/kaggle/working/{ds_name.upper()}/{adv_type}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)

    # Create ocsvm subfolder
    ocsvm_outf = f"{data_outf}/{ocsvm_fname}"
    if not os.path.isdir(ocsvm_outf):
        os.mkdir(ocsvm_outf)

    torch.cuda.manual_seed(0)
    torch.cuda.set_device(0)
    np.random.seed(0)

    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)

    fhandler = logging.FileHandler(f'{ocsvm_outf}/scan.log', 'a+', 'utf-8')
    shandler = logging.StreamHandler()

    formatter = logging.Formatter('%(asctime)s: %(message)s', datefmt="%Y-%m-%d %H:%M")
    fhandler.setFormatter(formatter)
    shandler.setFormatter(formatter)

    logger.addHandler(fhandler)
    logger.addHandler(shandler)

    logger.disables = not use_logger

    exp_name = f"{net_type}_{ds_name}_{adv_type}"
    logger.info(f"{exp_name}: STARTED")

    model, in_transform = get_model_transforms(net_type, ds_name, n_classes(ds_name))
    ds = DatasetsGA(ds_name, in_transform, net_type, adv_type, outf)

    clf = Pipeline(steps=[
        ('scaler', GroupedScaler()),
        ('PCA', PCA(whiten=True)),
        ('clf', OneClassSVM(kernel='rbf'))])

    if pre_computed:
        layer_det = clf
    else:
        split = PredefinedSplit([-1]*len(ds.train_loader.dataset) + [1]*len(ds.idxs_val))
        search_spaces = {
            "clf__nu": Real(2**-7, 2**-1, prior='log-uniform', base=2),
            "clf__gamma": Real(2**-15, 2**5, prior='log-uniform', base=2)
        }

        # Bayesian hyperparameter optimization
        layer_det = BayesSearchCV(clf, search_spaces, n_iter=bayes_n_iter, n_points=bayes_n_points,
            n_jobs=-1, scoring='accuracy', cv=split, return_train_score=False, refit=False, verbose=0)
        
    osvm_trainer = NetDetector(n_layers(net_type), layer_det, tqdm=False, logger=logger, exp_name=exp_name, outf=ocsvm_outf, pre_computed=pre_computed)
    osvm_trainer.fit(TrainValLoader(model, ds, batch_size=batch_size), LabelledTrainLoader(model, ds, batch_size=batch_size), ds.adv_test[ds.idxs_train])

    test_scores, output = osvm_trainer.predict(LabelledTestLoader(model, ds, batch_size=batch_size))

    all_output = np.hstack((test_scores, output, np.expand_dims(ds.adv_test[ds.idxs_test], axis=1)))

    with open(f'{ocsvm_outf}/OCSVM_net_detector_{exp_name}.pkl', 'wb') as outp:
        pickle.dump(osvm_trainer, outp, pickle.HIGHEST_PROTOCOL)

    np.save(f"{ocsvm_outf}/OCSVM_{exp_name}", all_output)

    accuracy = accuracy_score(ds.adv_test[ds.idxs_test], output[:,0])
    auroc = roc_auc_score(ds.adv_test[ds.idxs_test], -output[:,1])

    logger.info(f"{exp_name}: ACC = {accuracy}, AUROC = {round(auroc*100, 2)}")

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from skopt import BayesSearchCV
from skopt.space import Integer
import pickle
import os

from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
from skopt.space import Integer, Real

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm as tqdm_auto
import numpy as np
import pickle
import json

class SupervisedDetector:
    def __init__(self, n_layers, detector_class, tqdm=True, logger=None, exp_name="",
                 outf="", pre_computed=True, pre_computed_path=None):
        
        self.detector_class = detector_class
        self.tqdm = not tqdm
        self.n_layers = n_layers
        self.logger = logger
        self.exp_name = exp_name
        self.outf = outf
        self.pre_computed = pre_computed
        self.pre_computed_path = pre_computed_path

    def fit(self, dl_train, dl_unseen_train, adv_unseen_train):
        self.logger.info(f"{self.exp_name}: training layer detectors...")
        self.detectors = self.train_layer_detectors(dl_train)
        
        train_scores = self.get_layer_scores(dl_unseen_train)
        self.logger.info(f"{self.exp_name}: training final logistic...")
        self.lr = self.train_logistic_regression(train_scores, adv_unseen_train)

    def predict(self, data_loader):
        predicted_scores = self.get_layer_scores(data_loader)
        metrics = self.get_final_metrics(predicted_scores)
        return predicted_scores, metrics
        
    def train_layer_detectors(self, data_loader):
        detectors = []
        for layer_idx in tqdm_auto(range(self.n_layers), disable=not self.tqdm, 
                                   desc="Training layer detectors..."):
            self.logger.info(f"{self.exp_name}: started layer {layer_idx}")
            X_train, X_valid, y_train, y_valid, adv_train, adv_valid = data_loader(layer_idx)
            
            # Combine train and validation data
            X = np.concatenate((X_train, X_valid))
            y = np.concatenate((y_train, y_valid))
            adv = np.concatenate((adv_train, adv_valid))
            
            # Debug: Print unique classes and counts in adv
            unique, counts = np.unique(adv, return_counts=True)
            print(f"Layer {layer_idx} adv label distribution: {dict(zip(unique, counts))}")
            
            if self.pre_computed and self.pre_computed_path:
                # Load pre-computed parameters
                with open(self.pre_computed_path, "r") as fname:
                    params = json.load(fname)
                detector = clone(self.detector_class)
                best_params = params.get(f"{self.exp_name}_{layer_idx}", {})
                detector = detector.set_params(**best_params).fit(np.c_[X, y], adv)
            else:
                # Perform hyperparameter search with stratified cross-validation
                srch = clone(self.detector_class)
                
                # Set up stratified k-fold cross-validation
                stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                srch.set_params(cv=stratified_cv)
                
                # Fit with combined features and adversarial labels
                srch.fit(np.c_[X, y], adv)
                
                # Save the search results
                with open(f"{self.outf}/bayes_{self.exp_name}_{layer_idx}.pkl", "wb") as outp:
                    pickle.dump(srch, outp, pickle.HIGHEST_PROTOCOL)
                    
                # Train final detector with best parameters
                detector = srch.estimator.set_params(**srch.best_params_).fit(np.c_[X, y], adv)
                self.logger.info(f"{self.exp_name}: ACC = {srch.best_score_}, BEST_PARAMS = {srch.best_params_}")
            
            detectors.append(detector)
        return detectors

    def get_layer_scores(self, data_loader):
        scores = []
        for layer_idx in tqdm_auto(range(self.n_layers), disable=not self.tqdm, 
                                   desc="Extracting layer scores...", leave=False):
            X, y = data_loader(layer_idx)
            # Get probability predictions
            proba = self.detectors[layer_idx].predict_proba(np.c_[X, y])
            
            # Check what classes the detector learned
            classes = self.detectors[layer_idx].classes_
            #print(f"Layer {layer_idx} detector classes: {classes}")
            
            # Find the index corresponding to adversarial samples (class 0)
            if 0 in classes:
                adv_idx = np.where(classes == 0)[0][0]
            else:
                # Fallback to first class
                adv_idx = 0
                
            #print(f"Using index {adv_idx} for adversarial probability")
            scores.append(proba[:, adv_idx])  # Probability of adversarial class (0)
        
        return np.vstack(scores).T

    def train_logistic_regression(self, X, adv):
        lr = LogisticRegressionCV(penalty="l1", solver="liblinear", max_iter=10000, n_jobs=-1)
        lr.fit(X, adv)
        return lr
        
    def get_final_metrics(self, X):
        preds = self.lr.predict(X)
        probas = self.lr.predict_proba(X)
        return np.array([preds, probas[:, 0]]).T

def run_knn(net_type, ds_name, adv_type, outf='/kaggle/input/attacked-pth-files', knn_fname='knn', bayes_n_points=1, bayes_n_iter=10, use_logger=True, batch_size=100, pre_computed=True):
    outf = outf + f"/{ds_name.upper()}/{adv_type}"
    data_outf = f"/kaggle/working/{ds_name.upper()}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    data_outf = f"/kaggle/working/{ds_name.upper()}/{adv_type}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    knn_outf = f"{data_outf}/{knn_fname}"
    if not os.path.isdir(knn_outf):
        os.mkdir(knn_outf)

    torch.cuda.manual_seed(0)
    torch.cuda.set_device(0)
    np.random.seed(0)

    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    fhandler = logging.FileHandler(f'{knn_outf}/scan.log', 'a+', 'utf-8')
    shandler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s: %(message)s', datefmt="%Y-%m-%d %H:%M")
    fhandler.setFormatter(formatter)
    shandler.setFormatter(formatter)
    logger.addHandler(fhandler)
    logger.addHandler(shandler)
    logger.disables = not use_logger

    exp_name = f"{net_type}_{ds_name}_{adv_type}"
    logger.info(f"{exp_name}: STARTED")

    model, in_transform = get_model_transforms(net_type, ds_name, n_classes(ds_name))
    ds = DatasetsGA(ds_name, in_transform, net_type, adv_type, outf)

    clf = Pipeline(steps=[
        ('scaler', GroupedScaler()),
        ('PCA', PCA(whiten=True)),
        ('clf', KNeighborsClassifier())])

    if pre_computed:
        layer_det = clf
    else:
        # REMOVED PredefinedSplit - StratifiedKFold will be set in SupervisedDetector
        search_spaces = {
            "clf__n_neighbors": Integer(1, 15),
            "clf__weights": ["uniform", "distance"]
        }
        layer_det = BayesSearchCV(clf, search_spaces, n_iter=bayes_n_iter, n_points=bayes_n_points,
            n_jobs=-1, scoring='accuracy', return_train_score=False, refit=False, verbose=0)

    knn_trainer = SupervisedDetector(n_layers(net_type), layer_det, tqdm=False, logger=logger, exp_name=exp_name, outf=knn_outf, pre_computed=pre_computed)
    knn_trainer.fit(TrainValLoader(model, ds, batch_size=batch_size), LabelledTrainLoader(model, ds, batch_size=batch_size), ds.adv_test[ds.idxs_train])

    test_scores, output = knn_trainer.predict(LabelledTestLoader(model, ds, batch_size=batch_size))
    all_output = np.hstack((test_scores, output, np.expand_dims(ds.adv_test[ds.idxs_test], axis=1)))

    with open(f'{knn_outf}/KNN_net_detector_{exp_name}.pkl', 'wb') as outp:
        pickle.dump(knn_trainer, outp, pickle.HIGHEST_PROTOCOL)

    np.save(f"{knn_outf}/KNN_{exp_name}", all_output)

    accuracy = accuracy_score(ds.adv_test[ds.idxs_test], output[:, 0])
    auroc = roc_auc_score(ds.adv_test[ds.idxs_test], -output[:, 1])

    logger.info(f"{exp_name}: ACC = {accuracy}, AUROC = {round(auroc*100, 2)}")

def run_randomforest(net_type, ds_name, adv_type, outf='/kaggle/input/attacked-pth-files', rf_fname='randomforest', bayes_n_points=1, bayes_n_iter=10, use_logger=True, batch_size=100, pre_computed=True):
    outf = outf + f"/{ds_name.upper()}/{adv_type}"
    data_outf = f"/kaggle/working/{ds_name.upper()}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    data_outf = f"/kaggle/working/{ds_name.upper()}/{adv_type}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    rf_outf = f"{data_outf}/{rf_fname}"
    if not os.path.isdir(rf_outf):
        os.mkdir(rf_outf)

    torch.cuda.manual_seed(0)
    torch.cuda.set_device(0)
    np.random.seed(0)

    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    fhandler = logging.FileHandler(f'{rf_outf}/scan.log', 'a+', 'utf-8')
    shandler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s: %(message)s', datefmt="%Y-%m-%d %H:%M")
    fhandler.setFormatter(formatter)
    shandler.setFormatter(formatter)
    logger.addHandler(fhandler)
    logger.addHandler(shandler)
    logger.disables = not use_logger

    exp_name = f"{net_type}_{ds_name}_{adv_type}"
    logger.info(f"{exp_name}: STARTED")

    model, in_transform = get_model_transforms(net_type, ds_name, n_classes(ds_name))
    ds = DatasetsGA(ds_name, in_transform, net_type, adv_type, outf)

    clf = Pipeline(steps=[
        ('scaler', GroupedScaler()),
        ('PCA', PCA(whiten=True)),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=0))
    ])

    if pre_computed:
        layer_det = clf
    else:
        # REMOVED PredefinedSplit - StratifiedKFold will be set in SupervisedDetector
        search_spaces = {
            "clf__n_estimators": Integer(50, 200),
            "clf__max_depth": Integer(3, 20)
        }
        layer_det = BayesSearchCV(clf, search_spaces, n_iter=bayes_n_iter, n_points=bayes_n_points,
            n_jobs=-1, scoring='accuracy', return_train_score=False, refit=False, verbose=0)

    rf_trainer = SupervisedDetector(n_layers(net_type), layer_det, tqdm=False, logger=logger, exp_name=exp_name, outf=rf_outf, pre_computed=pre_computed)
    rf_trainer.fit(TrainValLoader(model, ds, batch_size=batch_size), LabelledTrainLoader(model, ds, batch_size=batch_size), ds.adv_test[ds.idxs_train])

    test_scores, output = rf_trainer.predict(LabelledTestLoader(model, ds, batch_size=batch_size))
    all_output = np.hstack((test_scores, output, np.expand_dims(ds.adv_test[ds.idxs_test], axis=1)))

    with open(f'{rf_outf}/RF_net_detector_{exp_name}.pkl', 'wb') as outp:
        pickle.dump(rf_trainer, outp, pickle.HIGHEST_PROTOCOL)

    np.save(f"{rf_outf}/RF_{exp_name}", all_output)

    accuracy = accuracy_score(ds.adv_test[ds.idxs_test], output[:, 0])
    auroc = roc_auc_score(ds.adv_test[ds.idxs_test], -output[:, 1])

    logger.info(f"{exp_name}: ACC = {accuracy}, AUROC = {round(auroc*100, 2)}")

def run_adaboost(net_type, ds_name, adv_type, outf='/kaggle/input/attacked-pth-files', ab_fname='adaboost', bayes_n_points=1, bayes_n_iter=10, use_logger=True, batch_size=100, pre_computed=True):
    outf = outf + f"/{ds_name.upper()}/{adv_type}"
    data_outf = f"/kaggle/working/{ds_name.upper()}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    data_outf = f"/kaggle/working/{ds_name.upper()}/{adv_type}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    ab_outf = f"{data_outf}/{ab_fname}"
    if not os.path.isdir(ab_outf):
        os.mkdir(ab_outf)

    torch.cuda.manual_seed(0)
    torch.cuda.set_device(0)
    np.random.seed(0)

    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    fhandler = logging.FileHandler(f'{ab_outf}/scan.log', 'a+', 'utf-8')
    shandler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s: %(message)s', datefmt="%Y-%m-%d %H:%M")
    fhandler.setFormatter(formatter)
    shandler.setFormatter(formatter)
    logger.addHandler(fhandler)
    logger.addHandler(shandler)
    logger.disabled = not use_logger

    exp_name = f"{net_type}_{ds_name}_{adv_type}"
    logger.info(f"{exp_name}: STARTED")

    model, in_transform = get_model_transforms(net_type, ds_name, n_classes(ds_name))
    ds = DatasetsGA(ds_name, in_transform, net_type, adv_type, outf)

    clf = Pipeline(steps=[
        ('scaler', GroupedScaler()),
        ('PCA', PCA(whiten=True)),
        ('clf', AdaBoostClassifier(n_estimators=100, random_state=0))
    ])

    if pre_computed:
        layer_det = clf
    else:
        search_spaces = {
            "clf__n_estimators": Integer(50, 200)
        }
        layer_det = BayesSearchCV(clf, search_spaces, n_iter=bayes_n_iter, n_points=bayes_n_points,
            n_jobs=-1, scoring='accuracy', return_train_score=False, refit=False, verbose=0)

    ab_trainer = SupervisedDetector(n_layers(net_type), layer_det, tqdm=False, logger=logger, exp_name=exp_name, outf=ab_outf, pre_computed=pre_computed)
    ab_trainer.fit(TrainValLoader(model, ds, batch_size=batch_size), LabelledTrainLoader(model, ds, batch_size=batch_size), ds.adv_test[ds.idxs_train])

    test_scores, output = ab_trainer.predict(LabelledTestLoader(model, ds, batch_size=batch_size))
    all_output = np.hstack((test_scores, output, np.expand_dims(ds.adv_test[ds.idxs_test], axis=1)))

    with open(f'{ab_outf}/AB_net_detector_{exp_name}.pkl', 'wb') as outp:
        pickle.dump(ab_trainer, outp, pickle.HIGHEST_PROTOCOL)

    np.save(f"{ab_outf}/AB_{exp_name}", all_output)

    accuracy = accuracy_score(ds.adv_test[ds.idxs_test], output[:, 0])
    auroc = roc_auc_score(ds.adv_test[ds.idxs_test], -output[:, 1])

    logger.info(f"{exp_name}: ACC = {accuracy}, AUROC = {round(auroc*100, 2)}")

def run_lightgbm(net_type, ds_name, adv_type, outf='/kaggle/input/attacked-pth-files', lgbm_fname='lightgbm', bayes_n_points=1, bayes_n_iter=10, use_logger=True, batch_size=100, pre_computed=True):
    outf = outf + f"/{ds_name.upper()}/{adv_type}"
    data_outf = f"/kaggle/working/{ds_name.upper()}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    data_outf = f"/kaggle/working/{ds_name.upper()}/{adv_type}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    lgbm_outf = f"{data_outf}/{lgbm_fname}"
    if not os.path.isdir(lgbm_outf):
        os.mkdir(lgbm_outf)

    torch.cuda.manual_seed(0)
    torch.cuda.set_device(0)
    np.random.seed(0)

    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    fhandler = logging.FileHandler(f'{lgbm_outf}/scan.log', 'a+', 'utf-8')
    shandler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s: %(message)s', datefmt="%Y-%m-%d %H:%M")
    fhandler.setFormatter(formatter)
    shandler.setFormatter(formatter)
    logger.addHandler(fhandler)
    logger.addHandler(shandler)
    logger.disabled = not use_logger

    exp_name = f"{net_type}_{ds_name}_{adv_type}"
    logger.info(f"{exp_name}: STARTED")

    model, in_transform = get_model_transforms(net_type, ds_name, n_classes(ds_name))
    ds = DatasetsGA(ds_name, in_transform, net_type, adv_type, outf)

    clf = Pipeline(steps=[
        ('scaler', GroupedScaler()),
        ('PCA', PCA(whiten=True)),
        ('clf', lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=0))
    ])

    if pre_computed:
        layer_det = clf
    else:
        search_spaces = {
            "clf__n_estimators": Integer(50, 200),
            "clf__max_depth": Integer(3, 10)
        }
        layer_det = BayesSearchCV(clf, search_spaces, n_iter=bayes_n_iter, n_points=bayes_n_points,
            n_jobs=-1, scoring='accuracy', return_train_score=False, refit=False, verbose=0)

    lgbm_trainer = SupervisedDetector(n_layers(net_type), layer_det, tqdm=False, logger=logger, exp_name=exp_name, outf=lgbm_outf, pre_computed=pre_computed)
    lgbm_trainer.fit(TrainValLoader(model, ds, batch_size=batch_size), LabelledTrainLoader(model, ds, batch_size=batch_size), ds.adv_test[ds.idxs_train])

    test_scores, output = lgbm_trainer.predict(LabelledTestLoader(model, ds, batch_size=batch_size))
    all_output = np.hstack((test_scores, output, np.expand_dims(ds.adv_test[ds.idxs_test], axis=1)))

    with open(f'{lgbm_outf}/LGBM_net_detector_{exp_name}.pkl', 'wb') as outp:
        pickle.dump(lgbm_trainer, outp, pickle.HIGHEST_PROTOCOL)

    np.save(f"{lgbm_outf}/LGBM_{exp_name}", all_output)

    accuracy = accuracy_score(ds.adv_test[ds.idxs_test], output[:, 0])
    auroc = roc_auc_score(ds.adv_test[ds.idxs_test], -output[:, 1])

    logger.info(f"{exp_name}: ACC = {accuracy}, AUROC = {round(auroc*100, 2)}")

def run_xgboost(net_type, ds_name, adv_type, outf='/kaggle/input/attacked-pth-files', xgb_fname='xgboost', bayes_n_points=1, bayes_n_iter=10, use_logger=True, batch_size=100, pre_computed=True):
    outf = outf + f"/{ds_name.upper()}/{adv_type}"
    data_outf = f"/kaggle/working/{ds_name.upper()}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    data_outf = f"/kaggle/working/{ds_name.upper()}/{adv_type}"
    if not os.path.isdir(data_outf):
        os.mkdir(data_outf)
    xgb_outf = f"{data_outf}/{xgb_fname}"
    if not os.path.isdir(xgb_outf):
        os.mkdir(xgb_outf)

    torch.cuda.manual_seed(0)
    torch.cuda.set_device(0)
    np.random.seed(0)

    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    fhandler = logging.FileHandler(f'{xgb_outf}/scan.log', 'a+', 'utf-8')
    shandler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s: %(message)s', datefmt="%Y-%m-%d %H:%M")
    fhandler.setFormatter(formatter)
    shandler.setFormatter(formatter)
    logger.addHandler(fhandler)
    logger.addHandler(shandler)
    logger.disables = not use_logger

    exp_name = f"{net_type}_{ds_name}_{adv_type}"
    logger.info(f"{exp_name}: STARTED")

    model, in_transform = get_model_transforms(net_type, ds_name, n_classes(ds_name))
    ds = DatasetsGA(ds_name, in_transform, net_type, adv_type, outf)

    clf = Pipeline(steps=[
        ('scaler', GroupedScaler()),
        ('PCA', PCA(whiten=True)),
        ('clf', xgb.XGBClassifier(n_estimators=100, max_depth=5))
    ])

    if pre_computed:
        layer_det = clf
    else:
        search_spaces = {
            "clf__n_estimators": Integer(50, 200),
            "clf__max_depth": Integer(3, 10)
        }
        layer_det = BayesSearchCV(clf, search_spaces, n_iter=bayes_n_iter, n_points=bayes_n_points,
            n_jobs=-1, scoring='accuracy', return_train_score=False, refit=False, verbose=0)

    xgb_trainer = SupervisedDetector(n_layers(net_type), layer_det, tqdm=False, logger=logger, exp_name=exp_name, outf=xgb_outf, pre_computed=pre_computed)
    xgb_trainer.fit(TrainValLoader(model, ds, batch_size=batch_size), LabelledTrainLoader(model, ds, batch_size=batch_size), ds.adv_test[ds.idxs_train])

    test_scores, output = xgb_trainer.predict(LabelledTestLoader(model, ds, batch_size=batch_size))
    all_output = np.hstack((test_scores, output, np.expand_dims(ds.adv_test[ds.idxs_test], axis=1)))

    with open(f'{xgb_outf}/XGB_net_detector_{exp_name}.pkl', 'wb') as outp:
        pickle.dump(xgb_trainer, outp, pickle.HIGHEST_PROTOCOL)

    np.save(f"{xgb_outf}/XGB_{exp_name}", all_output)

    accuracy = accuracy_score(ds.adv_test[ds.idxs_test], output[:, 0])
    auroc = roc_auc_score(ds.adv_test[ds.idxs_test], -output[:, 1])

    logger.info(f"{exp_name}: ACC = {accuracy}, AUROC = {round(auroc*100, 2)}")

def f1_scr(y_true, y_pred, pos_label=0):
    precision, recall, _ = precision_recall_curve(y_true, y_pred, pos_label=pos_label)
    f1_scores = 2*(precision * recall) / (precision + recall + 1e-8)
    return np.max(f1_scores)

def n_classes(ds_name):
    return 100 if ds_name == 'cifar100' else 10

def n_layers(net_type):
    return 5 if net_type == 'resnet' else 3

from PIL import Image

def compute_lid_for_single_image(image_tensor, model, ref_features, k=20, n_layers=5):
    lid_scores = []
    with torch.no_grad():
        _, features = model.feature_list(image_tensor.cuda())
    
    for layer_idx, (feat, ref_feat) in enumerate(zip(features[:n_layers], ref_features[:n_layers])):
        if len(feat.shape) > 2:
            feat = feat.view(feat.size(0), feat.size(1), -1).mean(dim=2)
        feat_flat = feat.cpu().numpy().reshape(1, -1)
        ref_flat = ref_feat.reshape(ref_feat.shape[0], -1)
        
        if feat_flat.shape[1] != ref_flat.shape[1]:
            raise ValueError(f"Feature dimension mismatch at layer {layer_idx}: "
                             f"image feature {feat_flat.shape[1]} vs ref feature {ref_flat.shape[1]}")
        
        nbrs = NearestNeighbors(n_neighbors=k).fit(ref_flat)
        distances, _ = nbrs.kneighbors(feat_flat)
        lid = -np.mean(np.log(distances + 1e-10), axis=1)
        lid_scores.append(lid)
    
    return np.concatenate(lid_scores)

from sklearn.neighbors import NearestNeighbors

def train_model(ds_name, net_type, adv_type, outf, binary_list, path_to_save_model, state):
    if len(binary_list) != 8 or not all(isinstance(x, int) and x in [0, 1] for x in binary_list):
        raise ValueError("binary_list must be a list of 8 integers, each 0 or 1")

    if all(x == 0 for x in binary_list):
        print("All detectors disabled, returning zero metrics")
        return (0, 0, 0)

    feature_sources = [
        ('lid', 'LID', 0, lambda: np.load(f"/kaggle/input/best-numpy-new/best numpy new/best {ds_name}/LID_best_{ds_name}_{adv_type}.npy")),
        ('mahalanobis', 'Mahalanobis', 1, lambda: np.load(f"/kaggle/input/best-numpy-new/best numpy new/best {ds_name}/Mahalanobis_best_{ds_name}_{adv_type}.npy")),
        ('ocsvm', 'OCSVM', 2, lambda: (
            pkl.load(open(f'/kaggle/input/ocsvm-new-pkl/ocsvm pkl new/{ds_name}/{adv_type}/OCSVM_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            np.load(f"/kaggle/input/ocsvm-new-pkl/ocsvm pkl new/{ds_name}/{adv_type}/OCSVM_{net_type}_{ds_name}_{adv_type}.npy")
        )),
        ('knn', 'KNN', 3, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/KNN/{adv_type}/KNN_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            np.load(f"/kaggle/input/new-ratio-full-pkl/{ds_name}/KNN/{adv_type}/KNN_{net_type}_{ds_name}_{adv_type}.npy")
        )),
        ('randomforest', 'RF', 4, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/Random Forest/{adv_type}/RF_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            np.load(f"/kaggle/input/new-ratio-full-pkl/{ds_name}/Random Forest/{adv_type}/RF_{net_type}_{ds_name}_{adv_type}.npy")
        )),
        ('adaboost', 'AB', 5, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/AdaBoost/{adv_type}/AB_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            np.load(f"/kaggle/input/new-ratio-full-pkl/{ds_name}/AdaBoost/{adv_type}/AB_{net_type}_{ds_name}_{adv_type}.npy")
        )),
        ('xgboost', 'XGB', 6, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/XGBoost/{adv_type}/XGB_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            np.load(f"/kaggle/input/new-ratio-full-pkl/{ds_name}/XGBoost/{adv_type}/XGB_{net_type}_{ds_name}_{adv_type}.npy")
        )),
        ('lightgbm', 'LGBM', 7, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/LightGBM/{adv_type}/LGBM_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            np.load(f"/kaggle/input/new-ratio-full-pkl/{ds_name}/LightGBM/{adv_type}/LGBM_{net_type}_{ds_name}_{adv_type}.npy")
        )),
    ]

    model, in_transform = get_model_transforms(net_type, ds_name, n_classes(ds_name))
    dataset = DatasetsGA(ds_name, in_transform, net_type, adv_type, 
                        f"/kaggle/input/attacked-pth-files/{ds_name.upper()}/{adv_type}")
    data_loader = LabelledTrainLoader(model, dataset)
    eval_loader = LabelledGAValLoader(model, dataset)
    eval_idxs = dataset.idxs_ga_val
    n_layers_used = n_layers(net_type)

    X_train_all = []
    X_eval_all = []

    for fname, prefix, idx, load_fn in feature_sources:
        if binary_list[idx] == 1:
            try:
                if fname in ['lid', 'mahalanobis']:
                    data = load_fn()
                    X_train_all.append(data[dataset.idxs_train][:, :n_layers_used])
                    X_eval_all.append(data[eval_idxs][:, :n_layers_used])
                else:
                    det, det_test = load_fn()
                    det_train_data = det.get_layer_scores(data_loader)
                    det_eval_data = det.get_layer_scores(eval_loader)
                    X_train_all.append(det_train_data[:, :n_layers_used])
                    X_eval_all.append(det_eval_data[:, :n_layers_used])
            except Exception as e:
                print(f"Error loading {prefix}: {e}")
                continue

    if not X_train_all or not X_eval_all:
        raise ValueError("No features were successfully loaded based on the binary list")

    X_train = np.c_[tuple(X_train_all)]
    X_eval = np.c_[tuple(X_eval_all)]

    lr = LogisticRegressionCV(penalty='l1', solver='liblinear', max_iter=7000, n_jobs=-1)
    lr.fit(X_train, dataset.adv_test[dataset.idxs_train])

    adv_conf_eval = lr.predict_proba(X_eval)[:, 0]
    auroc_eval = roc_auc_score(dataset.adv_test[eval_idxs], -adv_conf_eval)
    aupr_eval = aupr(dataset.adv_test[eval_idxs], adv_conf_eval, pos_label=0)
    f1_test = f1_scr(dataset.adv_test[eval_idxs], adv_conf_eval, pos_label=0)

    print(f"AUROC: {auroc_eval:.4f}, AUPR: {aupr_eval:.4f}, F1 Score: {f1_test:.4f}")

    # Save the logistic regression model
    model_path = f"{path_to_save_model}/{ds_name}_{adv_type}_{state}_lr.pkl"
    with open(model_path, 'wb') as f:
        pkl.dump(lr, f)

    # Save the training and evaluation features
    #np.save(f"{path_to_save_model}_train_features.npy", X_train)
    #np.save(f"{path_to_save_model}_eval_features.npy", X_eval)

    return auroc_eval, aupr_eval, f1_test

def predict_image(ds_name, net_type, adv_type, outf, binary_list, image_path, state):
    if len(binary_list) != 8 or not all(isinstance(x, int) and x in [0, 1] for x in binary_list):
        raise ValueError("binary_list must be a list of 8 integers, each 0 or 1")

    # Load the logistic regression model
    model_path = f"/kaggle/input/lr-pickle/{ds_name}_{adv_type}_{state}_lr.pkl"
    with open(model_path, 'rb') as f:
        lr = pkl.load(f)

    # Load model and transformations
    model, in_transform = get_model_transforms(net_type, ds_name, n_classes(ds_name))
    n_layers_used = n_layers(net_type)

    # Load and process the image
    image = Image.open(image_path).convert("RGB")
    image_tensor = in_transform(image).unsqueeze(0).cuda()

    # Feature sources for detectors
    feature_sources = [
        ('lid', 'LID', 0, lambda: pkl.load(open(f'/kaggle/input/stat-and-features/{ds_name}_{adv_type}_lid_training_features.pkl', 'rb'))),
        ('mahalanobis', 'Mahalanobis', 1, lambda: pkl.load(open(f'/kaggle/input/stat-and-features/{ds_name}_{adv_type}_mahalanobis_stats.pkl', 'rb'))),
        ('ocsvm', 'OCSVM', 2, lambda: (
            pkl.load(open(f'/kaggle/input/ocsvm-new-pkl/ocsvm pkl new/{ds_name}/{adv_type}/OCSVM_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            None
        )),
        ('knn', 'KNN', 3, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/KNN/{adv_type}/KNN_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            None
        )),
        ('randomforest', 'RF', 4, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/Random Forest/{adv_type}/RF_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            None
        )),
        ('adaboost', 'AB', 5, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/AdaBoost/{adv_type}/AB_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            None
        )),
        ('xgboost', 'XGB', 6, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/XGBoost/{adv_type}/XGB_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            None
        )),
        ('lightgbm', 'LGBM', 7, lambda: (
            pkl.load(open(f'/kaggle/input/new-ratio-full-pkl/{ds_name}/LightGBM/{adv_type}/LGBM_net_detector_{net_type}_{ds_name}_{adv_type}.pkl', 'rb')),
            None
        )),
    ]

    with torch.no_grad():
        _, features = model.feature_list(image_tensor)

    maha_feats = []
    lid_feats = []
    other_feats = []

    if binary_list[1]:
        maha_stats = feature_sources[1][3]()
        for i in range(n_layers_used):
            feat = features[i]
            if len(feat.shape) > 2:
                feat = feat.view(feat.size(0), feat.size(1), -1).mean(dim=2)
            feat_np = feat.cpu().numpy().reshape(-1)
            means, inv_covs = maha_stats
            mean = means[i].cpu()
            inv_cov = inv_covs[i].cpu()
            min_score = float('inf')
            for class_idx in range(mean.shape[0]):
                class_mean = mean[class_idx].cpu().numpy()
                score = mahalanobis(feat_np, class_mean, inv_cov.cpu().numpy())
                if score < min_score:
                    min_score = score
            maha_feats.append(min_score)

    if binary_list[0]:
        lid_ref = feature_sources[0][3]()
        for i in range(n_layers_used):
            feat = features[i]
            if len(feat.shape) > 2:
                feat = feat.view(feat.size(0), feat.size(1), -1).mean(dim=2)
            feat_np = feat.cpu().numpy().reshape(1, -1)
            ref_feat = lid_ref[i]
            nbrs = NearestNeighbors(n_neighbors=20).fit(ref_feat)
            distances, _ = nbrs.kneighbors(feat_np)
            lid_score = -np.mean(np.log(distances + 1e-10))
            lid_feats.append(lid_score)

    other_detector_names = ['ocsvm', 'knn', 'randomforest', 'adaboost', 'xgboost', 'lightgbm']
    for idx, (fname, prefix, feature_idx, load_fn) in enumerate(feature_sources[2:], start=2):
        if binary_list[feature_idx] == 1:
            detector, _ = load_fn()
            class SingleImageLoader:
                def __init__(self, image_tensor, model, n_layers_used, label=0):
                    self.image_tensor = image_tensor
                    self.label = label
                    self.n_layers_used = n_layers_used
                    with torch.no_grad():
                        _, features = model.feature_list(image_tensor)
                    self.features = []
                    for i in range(n_layers_used):
                        feat = features[i]
                        if len(feat.shape) > 2:
                            feat = feat.view(feat.size(0), feat.size(1), -1).mean(dim=2)
                        feat_np = feat.cpu().numpy().reshape(-1)
                        self.features.append(feat_np)

                def __call__(self, layer_idx):
                    feat_vector = self.features[layer_idx].reshape(1, -1)
                    label = np.array([self.label])
                    return feat_vector, label

            single_loader = SingleImageLoader(image_tensor, model, n_layers_used)
            scores = detector.get_layer_scores(single_loader)
            other_feats.append(scores[0, :n_layers_used])

    all_feats_list = []
    if binary_list[0]:
        all_feats_list.append(np.array(lid_feats).reshape(1, -1))
    if binary_list[1]:
        all_feats_list.append(np.array(maha_feats).reshape(1, -1))
    if other_feats:
        all_feats_list.append(np.hstack(other_feats).reshape(1, -1))

    X_img = np.hstack(all_feats_list)
    label = lr.predict(X_img)[0]
    label_str = 'adversarial' if label == 0 else 'normal'

    print(f"Predicted Label: {label_str}")
    return label_str

import streamlit as st
from PIL import Image
import pickle as pkl
from scipy.spatial.distance import mahalanobis
from sklearn.neighbors import NearestNeighbors
import io
import sys
from contextlib import redirect_stdout, redirect_stderr

def main():
    st.title("Adversarial Image Detection")
    st.write("Upload an image and select options to predict if it's adversarial or normal.")

    uploaded_file = st.file_uploader("Upload an image", type=["png", "jpg", "jpeg"])

    ds_name = st.selectbox("Select Dataset", ["cifar10", "svhn"])
    adv_type = st.selectbox("Select Adversarial Attack Type", ["FGSM", "BIM", "DeepFool", "CWL2"])
    algorithm = st.selectbox("Select Algorithm", ["enad", "enad_ga"])

    net_type = 'resnet'
    outf = '/kaggle/working'

    if algorithm == "enad":
        binary_list = [1, 1, 1, 0, 0, 0, 0, 0]
    else:  
        if ds_name == "cifar10":
            binary_list = {
                "FGSM": [0, 1, 1, 0, 1, 1, 1, 0],
                "BIM": [0, 0, 0, 0, 0, 1, 0, 1],
                "DeepFool": [1, 1, 1, 1, 1, 0, 1, 0],
                "CWL2": [0, 0, 1, 1, 1, 1, 1, 0]
            }[adv_type]
        else:  # svhn
            binary_list = {
                "FGSM": [1, 0, 0, 1, 1, 0, 0, 0],
                "BIM": [1, 0, 0, 1, 1, 1, 1, 0],
                "DeepFool": [1, 1, 1, 1, 1, 1, 1, 0],
                "CWL2": [1, 0, 0, 0, 0, 1, 0, 1]
            }[adv_type]

    # Predict button
    if st.button("Predict"):
        if uploaded_file is None:
            st.error("Please upload an image.")
        else:
            try:
                with open("temp_image.png", "wb") as f:
                    f.write(uploaded_file.getbuffer())

                with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                    label = predict_image(ds_name, net_type, adv_type, outf, binary_list, "temp_image.png", algorithm)

                st.success(f"Predicted Label: {label}")

                st.image(uploaded_file, caption="Uploaded Image", use_column_width=True)

            except Exception as e:
                st.error(f"Error during prediction: {str(e)}")

main()

Writing app_temp.py


In [4]:
!wget -q -O - ipv4.icanhazip.com

34.83.121.121


In [ ]:
!streamlit run app_temp.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴

your url is: https://fruity-stars-marry.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.83.121.121:8501

2025-06-11 14:41:04.132 Examining the path of torch.classes raised:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/streamlit/web/bootstrap.py", line 347, in run
    if asyncio.get_running_loop().is_running():
RuntimeError: no running event loop

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/streamlit/watcher/local_sources_watcher.py", line 217, in get_module_paths
    potential_paths = extract_paths(module)
  File "/usr/local/lib/python3.10/dist-packages/streamlit/watcher/local_sources_watcher.py", line 210, in <lambda>
    lambda m: list(m.__path__._path),
  File "/usr/local/lib/python3.10/dist-packages/torch/_classes.py"